[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/12_llm_agents_recsys/12_llm_agents_recsys.ipynb)

# Módulo 12 · LLMs y agentes en sistemas de recomendación

**Nivel:** 🔴 Experto · **Duración:** 7–9 h · **GPU recomendada:** L4 (24 GB) para modelos 7–8B en bf16; T4 sirve con cuantización 4-bit o modelos 1,5–3B; A100 si haces el fine-tuning LoRA a 7B. Sin GPU el notebook funciona entero con una API (Claude/OpenAI) o con el backend `mock`.
**Unidades de Colab estimadas:** ~15–25 en L4 (lo caro es el fine-tuning LoRA y el ranking con 7B); ~0 en modo API/mock.
**Prerrequisitos:** 01 (datos y splits), 02 (métricas), 03 (embeddings de contenido), 08 (retrieval + FAISS). Recomendado: 09 (secuencial) y 11 (generativos).

> Ya conoces LangChain/LangGraph y los agentes. Aquí no repetimos lo básico: vamos a **lo que cambia cuando el agente tiene que recomendar** — catálogos de miles de ítems que el LLM no conoce, latencias de milisegundos, sesgos de posición, alucinaciones de ítems que no existen y evaluación sin humanos.

## 🎯 Objetivos de aprendizaje

Al terminar este módulo serás capaz de:

1. **Usar un LLM como encoder** de ítems y medir cuándo los embeddings de texto superan a los IDs (cold start) y cuándo no.
2. **Implementar un re-ranker LLM listwise** (zero-shot) robusto: prompt, parser anti-alucinación, *bootstrapping* contra el sesgo de posición, y evaluarlo con NDCG sobre la salida real de un retriever.
3. **Ajustar un LLM con LoRA al estilo TALLRec** y compararlo honestamente con baselines clásicos (AUC).
4. **Enriquecer metadatos y resolver cold start** con LLMs offline (JSON validado, caché, coste) y mapear texto → espacio colaborativo.
5. **Construir un LLM-as-judge** con protocolo de intercambio de posiciones y medir su acuerdo con las métricas offline.
6. **Generar explicaciones fundamentadas** y detectar automáticamente las alucinadas.
7. **Diseñar un agente conversacional con LangGraph** (estado tipado, herramientas de recsys, memoria de preferencias a corto y largo plazo, *guardrails* de catálogo).
8. **Evaluar agentes recomendadores** con simuladores de usuario (Agent4Rec / iEvaLM) y estimar coste, latencia y tasa de alucinación en producción.

### Cómo está organizado

| § | Rol del LLM | Dónde vive en producción |
|---|---|---|
| 2 | **Encoder** de ítems/usuarios | offline / nearline (embeddings precomputados) |
| 3 | **Ranker** zero-shot listwise | online (caro) o como *teacher* offline |
| 4 | **Fine-tuning LoRA** (TALLRec) | offline (entrenamiento) |
| 5 | **Enriquecimiento** + cold start | offline batch |
| 6 | **Juez** (LLM-as-judge) | evaluación offline |
| 7 | **Explicador** | online (streaming) o precomputado |
| 8 | **Agente conversacional** (LangGraph) | online |
| 9–10 | **Simulador de usuarios** + evaluación de agentes | evaluación offline |
| 11 | Costes, latencia, alucinación | — |

**Backends del LLM.** Todo el notebook usa una interfaz `BaseLLM.generate()` con cinco implementaciones:
`hf` (🤗 transformers, p. ej. `Qwen/Qwen2.5-7B-Instruct`, `Qwen/Qwen3-8B`, `meta-llama/Llama-3.1-8B-Instruct`), `vllm` (el mismo modelo con batching continuo), `anthropic` (por defecto `claude-sonnet-5-5`, variable `ANTHROPIC_API_KEY`), `openai` (`OPENAI_API_KEY`) y `mock` (simulador sin GPU ni red para probar la tubería). Con `LLM_BACKEND="auto"` se elige GPU → API → mock.

> ⚠️ **Cuidado:** con el backend `mock` todo se ejecuta, pero **las cifras no son de un LLM**. Úsalo para entender el código y luego ejecuta con `hf` o `anthropic` para ver resultados reales.

In [ ]:
!pip install -q "sentence-transformers>=3" faiss-cpu "langgraph>=0.3" langchain-core langchain-anthropic anthropic openai "transformers>=4.45" accelerate peft bitsandbytes
# Opcional (L4/A100): vLLM para servir el LLM con batching continuo → !pip install -q vllm

In [ ]:
import os, re, json, time, math, random, operator, itertools, collections, warnings
from typing import TypedDict, Annotated
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch, device = None, "cpu"

# Claves de API desde los "Secrets" de Colab (icono 🔑), si existen
try:
    from google.colab import userdata
    for _k in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"]:
        try:
            _v = userdata.get(_k)
        except Exception:
            _v = None
        if _v:
            os.environ.setdefault(_k, _v)
except ImportError:
    pass

FAST_DEV_RUN = True                  # True: pocos usuarios/llamadas; False: escala completa en GPU
DATASET = "ml-latest-small"          # tiene títulos+géneros: ideal para prompts. Alternativa: "ml-1m"
# Backend del LLM: "auto" | "hf" (transformers) | "vllm" | "anthropic" | "openai" | "mock"
LLM_BACKEND = os.environ.get("LLM_BACKEND", "auto")
HF_MODEL = os.environ.get("HF_MODEL", "Qwen/Qwen2.5-7B-Instruct")   # o "Qwen/Qwen3-8B", "meta-llama/Llama-3.1-8B-Instruct"
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
EMB_MODEL = os.environ.get("EMB_MODEL", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
N_EVAL_USERS = 40 if FAST_DEV_RUN else 400
K = 10

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False})
print(f"device={device} | backend pedido={LLM_BACKEND} | FAST_DEV_RUN={FAST_DEV_RUN}")

## 0 · Utilidades mínimas (de los módulos 01, 02 y 08)

Para que el notebook sea autocontenido en Colab copiamos aquí, en versión mínima, lo que construiste antes:
- **Módulo 01:** descarga de MovieLens (con *fallback* sintético si no hay red) y split temporal por usuario.
- **Módulo 02:** `Recall@K` y `NDCG@K` (relevancia binaria).
- **Módulo 08:** retriever por producto interno con **FAISS** (`IndexFlatIP`). Como "two-tower ligero" usamos **PureSVD** (Cremonesi et al., 2010): $\hat{\mathbf{r}}_u = \mathbf{x}_u V V^\top$, que se entrena en segundos y es un baseline sorprendentemente fuerte.

In [ ]:
# ==== Utilidades mínimas (vienen del módulo 01: datos) ====
import io, zipfile, urllib.request
from pathlib import Path

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
ML_URLS = {
    "ml-latest-small": "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip",
    "ml-1m": "https://files.grouplens.org/datasets/movielens/ml-1m.zip",
}
ML_GENRES = ["Action", "Adventure", "Animation", "Children", "Comedy", "Crime", "Documentary",
             "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
             "Sci-Fi", "Thriller", "War", "Western"]

def _download_movielens(variant: str) -> Path:
    zpath = DATA_DIR / f"{variant}.zip"
    if not zpath.exists():
        urllib.request.urlretrieve(ML_URLS[variant], zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA_DIR)
    return DATA_DIR / variant

def make_synthetic_movielens(n_users=600, n_items=1500, seed=42):
    """Fallback sintético con la MISMA forma que MovieLens (por si no hay red)."""
    rng = np.random.default_rng(seed)
    words = {g: [f"{g[:4]}{k}" for k in range(6)] for g in ML_GENRES}
    rows_m = []
    for i in range(n_items):
        gs = list(rng.choice(ML_GENRES, size=rng.integers(1, 4), replace=False))
        year = int(rng.integers(1950, 2019))
        title = " ".join(rng.choice(words[gs[0]], 2)) + f" {i} ({year})"
        rows_m.append((i, title, "|".join(gs)))
    movies = pd.DataFrame(rows_m, columns=["item_id", "title", "genres"])
    G = np.array([[g in m.split("|") for g in ML_GENRES] for m in movies.genres], float)
    pop = rng.zipf(1.6, n_items).clip(1, 500).astype(float)
    rows_r = []
    for u in range(n_users):
        taste = rng.dirichlet(np.full(len(ML_GENRES), 0.2))
        aff = G @ taste / G.sum(1)
        p = pop ** 0.6 * np.exp(10 * aff); p /= p.sum()
        n = int(rng.integers(20, 150))
        items = rng.choice(n_items, size=n, replace=False, p=p)
        rat = np.clip(np.round(2.5 + 8 * aff[items] + rng.normal(0, .7, n)), 1, 5)
        ts = np.sort(rng.integers(9e8, 1.5e9, n))
        rows_r += list(zip([u] * n, items, rat, ts))
    ratings = pd.DataFrame(rows_r, columns=["user_id", "item_id", "rating", "timestamp"])
    return ratings, movies

In [ ]:
def load_movielens(variant: str = "ml-latest-small"):
    """Devuelve (ratings[user_id,item_id,rating,timestamp], movies[item_id,title,genres,year]) con ids contiguos."""
    try:
        d = _download_movielens(variant)
        if variant == "ml-1m":
            ratings = pd.read_csv(d / "ratings.dat", sep="::", engine="python",
                                  names=["user_id", "item_id", "rating", "timestamp"])
            movies = pd.read_csv(d / "movies.dat", sep="::", engine="python", encoding="latin-1",
                                 names=["item_id", "title", "genres"])
        else:
            ratings = pd.read_csv(d / "ratings.csv").rename(columns={"userId": "user_id", "movieId": "item_id"})
            movies = pd.read_csv(d / "movies.csv").rename(columns={"movieId": "item_id"})
        print(f"MovieLens {variant} descargado de GroupLens")
    except Exception as e:  # sin red / URL caída → nunca bloqueamos el notebook
        print(f"⚠️ No se pudo descargar MovieLens ({type(e).__name__}). Uso datos SINTÉTICOS con la misma forma.")
        ratings, movies = make_synthetic_movielens()
    movies = movies[movies.item_id.isin(ratings.item_id.unique())].reset_index(drop=True)
    item_map = {old: new for new, old in enumerate(movies.item_id)}
    user_map = {old: new for new, old in enumerate(sorted(ratings.user_id.unique()))}
    ratings = ratings.assign(item_id=ratings.item_id.map(item_map), user_id=ratings.user_id.map(user_map))
    movies = movies.assign(item_id=movies.item_id.map(item_map))
    movies["year"] = movies.title.str.extract(r"\((\d{4})\)\s*$")[0].astype(float)
    movies["genre_list"] = movies.genres.str.replace("Children's", "Children").str.split("|")   # ml-1m → mismo nombre
    return ratings.sort_values(["user_id", "timestamp"]).reset_index(drop=True), movies

In [ ]:
# ==== Utilidades mínimas (vienen de los módulos 01 y 02: split temporal + métricas) ====
def split_temporal_per_user(ratings: pd.DataFrame, test_frac: float = 0.2, min_train: int = 5):
    """Para cada usuario, el último `test_frac` de sus interacciones (por tiempo) va a test."""
    r = ratings.sort_values(["user_id", "timestamp"]).copy()
    r["rank_t"] = r.groupby("user_id").cumcount()
    r["n_u"] = r.groupby("user_id")["item_id"].transform("size")
    n_test = np.maximum(1, np.floor(r["n_u"] * test_frac)).astype(int)
    is_test = (r["rank_t"] >= r["n_u"] - n_test) & (r["n_u"] > min_train)
    return r[~is_test].drop(columns=["rank_t", "n_u"]), r[is_test].drop(columns=["rank_t", "n_u"])

def recall_at_k(recs, truth, k=10):
    return len(set(recs[:k]) & truth) / max(1, min(k, len(truth)))

def ndcg_at_k(recs, truth, k=10):
    dcg = sum(1 / np.log2(i + 2) for i, it in enumerate(recs[:k]) if it in truth)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(k, len(truth))))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(recs_by_user: dict, truth_by_user: dict, k: int = 10) -> dict:
    us = [u for u in recs_by_user if u in truth_by_user and truth_by_user[u]]
    return {f"Recall@{k}": float(np.mean([recall_at_k(recs_by_user[u], truth_by_user[u], k) for u in us])),
            f"NDCG@{k}": float(np.mean([ndcg_at_k(recs_by_user[u], truth_by_user[u], k) for u in us]))}

In [ ]:
# ==== Utilidades mínimas (vienen del módulo 08: retriever de embeddings + FAISS) ====
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
try:
    import faiss
except ImportError:
    faiss = None

def build_interaction_matrix(train: pd.DataFrame, n_users: int, n_items: int):
    return sp.csr_matrix((np.ones(len(train), np.float32), (train.user_id, train.item_id)),
                         shape=(n_users, n_items))

class EmbeddingRetriever:
    """Top-K por producto interno con FAISS (IndexFlatIP); fallback NumPy si no hay FAISS."""
    def __init__(self, item_emb: np.ndarray):
        self.item_emb = np.ascontiguousarray(item_emb, dtype=np.float32)
        self.index = None
        if faiss is not None:
            self.index = faiss.IndexFlatIP(self.item_emb.shape[1]); self.index.add(self.item_emb)

    def search(self, q: np.ndarray, k: int = 100, exclude=None):
        q = np.ascontiguousarray(np.atleast_2d(q), dtype=np.float32)
        extra = max((len(e) for e in exclude), default=0) if exclude is not None else 0
        kk = min(k + extra, len(self.item_emb))
        if self.index is not None:
            scores, ids = self.index.search(q, kk)
        else:
            s = q @ self.item_emb.T
            ids = np.argsort(-s, axis=1)[:, :kk]; scores = np.take_along_axis(s, ids, 1)
        out = []
        for row, (sc, ii) in enumerate(zip(scores, ids)):
            ex = exclude[row] if exclude is not None else set()
            keep = [(int(i), float(s)) for i, s in zip(ii, sc) if i >= 0 and i not in ex][:k]
            out.append(keep)
        return out

class PureSVDRecommender:
    """CF ligero (PureSVD, Cremonesi 2010) como sustituto del two-tower del módulo 08."""
    def __init__(self, X: sp.csr_matrix, dim: int = 64, seed: int = 42):
        self.X = X
        svd = TruncatedSVD(n_components=dim, random_state=seed).fit(X)
        self.item_emb = svd.components_.T.astype(np.float32)      # V  (n_items × d)
        self.retriever = EmbeddingRetriever(self.item_emb)

    def user_emb(self, users):
        return np.asarray(self.X[users] @ self.item_emb)          # u = x_u V

    def recommend(self, users, k=10, exclude_seen=True):
        users = list(users)
        excl = [set(self.X[u].indices) for u in users] if exclude_seen else None
        res = self.retriever.search(self.user_emb(users), k, excl)
        return {u: [i for i, _ in r] for u, r in zip(users, res)}

In [ ]:
ratings, movies = load_movielens(DATASET)
n_users, n_items = ratings.user_id.nunique(), len(movies)
train, test = split_temporal_per_user(ratings, test_frac=0.2)
X = build_interaction_matrix(train, n_users, n_items)
pop = np.asarray(X.sum(0)).ravel()                               # popularidad en train
truth = test.groupby("user_id").item_id.apply(set).to_dict()      # ítems futuros por usuario
hist_by_user = train.groupby("user_id").item_id.apply(list).to_dict()   # en orden temporal
rating_lookup = dict(zip(zip(ratings.user_id, ratings.item_id), ratings.rating))
rng = np.random.default_rng(SEED)
eval_users = [int(u) for u in rng.choice(sorted(truth), size=min(N_EVAL_USERS, len(truth)), replace=False)]
print(f"{n_users} usuarios · {n_items} películas · {len(ratings):,} ratings · train={len(train):,} test={len(test):,}")
movies.head(3)

## 1 · 💡 Intuición: ¿qué pinta un LLM en un recomendador?

Un recomendador industrial es un embudo: **retrieval** (millones → cientos) → **ranking** (cientos → decenas) → **re-ranking / políticas** → UI. Un LLM puede entrar en casi cualquier punto, pero con costes muy distintos: generar 100 tokens con un 7B cuesta ~100 pasadas del modelo; un ranker DCN-v2 puntúa miles de ítems en el mismo tiempo.

La taxonomía de Lin et al. (2023, *How Can Recommender Systems Benefit from LLMs*) pregunta **dónde** (ingeniería de features, encoder, función de ranking, interacción con el usuario, controlador del pipeline) y **cómo** (con o sin fine-tuning, con o sin un modelo convencional al lado). Wu et al. (2023) los agrupan en **discriminativos** (el LLM produce representaciones o puntuaciones) y **generativos** (el LLM genera la recomendación como texto o IDs, que viste en el módulo 11).

> 💡 **Analogía para un ML engineer:** el LLM es un *feature extractor* y un *labeler* carísimo pero con conocimiento del mundo. Lo rentable casi siempre es usarlo **offline** (embeddings, metadatos, etiquetas, simulación) y destilar; lo caro es ponerlo en el camino crítico de cada petición.

In [ ]:
def draw_llm_roles():
    fig, ax = plt.subplots(figsize=(12, 5.2)); ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(0, 6)
    stages = [("Catálogo\n+ logs", .3), ("Retrieval\n(FAISS)", 2.6), ("Ranking", 4.9), ("Re-ranking\n+ reglas", 7.2), ("UI / chat", 9.5)]
    for name, x in stages:
        ax.add_patch(plt.Rectangle((x, 3.6), 1.9, 1.1, fc="#dbe9f6", ec="#2b6cb0", lw=1.5))
        ax.text(x + .95, 4.15, name, ha="center", va="center", fontsize=10, weight="bold")
    for (_, x1), (_, x2) in zip(stages[:-1], stages[1:]):
        ax.annotate("", (x2, 4.15), (x1 + 1.9, 4.15), arrowprops=dict(arrowstyle="->", lw=1.6))
    roles = [("§5 Enriquecer\nmetadatos (offline)", 1.25, "#fde2b5"), ("§2 Encoder\n(embeddings)", 3.55, "#c6f0c2"),
             ("§3-4 Ranker LLM\n(zero-shot / LoRA)", 5.85, "#f9c6c6"), ("§7 Explicaciones", 8.15, "#e3d4f7"),
             ("§8 Agente\nconversacional", 10.45, "#ffd6e8")]
    for txt, x, c in roles:
        ax.add_patch(plt.Rectangle((x - .95, 1.7), 1.9, 1.1, fc=c, ec="gray"))
        ax.text(x, 2.25, txt, ha="center", va="center", fontsize=8.5)
        ax.annotate("", (x, 3.6), (x, 2.8), arrowprops=dict(arrowstyle="->", color="gray", ls="--"))
    ax.add_patch(plt.Rectangle((.3, .2), 11.1, 1.0, fc="#f1f1f1", ec="gray"))
    ax.text(5.85, .7, "Evaluación offline: §6 LLM-as-judge · §9 simulación de usuarios (Agent4Rec) · §10 evaluación de agentes",
            ha="center", va="center", fontsize=9.5)
    ax.text(.3, 5.4, "Dónde entra un LLM en el embudo de recomendación", fontsize=13, weight="bold")
    plt.show()
draw_llm_roles()

La regla de oro que vas a comprobar con números en §11:

| Uso | Llamadas al LLM | Escala |
|---|---|---|
| Enriquecer el catálogo | 1 por ítem (una vez) | $O(\lvert\mathcal{I}\rvert)$ — barato |
| Embeddings de ítems | 1 por ítem | $O(\lvert\mathcal{I}\rvert)$ — barato |
| Re-ranking online | 1 por petición | $O(\text{QPS})$ — caro |
| Agente conversacional | varias por turno | $O(\text{sesiones} \times \text{turnos})$ — caro, pero solo para quien conversa |

## 2 · El LLM como encoder: embeddings de texto para ítems

### 📐 Teoría
Un modelo de lenguaje $f_\theta$ convierte el texto de un ítem $t_i$ (título, géneros, sinopsis…) en un vector:
$$\mathbf{e}_i = \frac{\operatorname{pool}(f_\theta(t_i))}{\lVert \operatorname{pool}(f_\theta(t_i)) \rVert} \in \mathbb{R}^d$$
donde `pool` es *mean pooling* (sentence-transformers) o el último token (encoders basados en decoder como **Qwen3-Embedding** o **e5-mistral**). Con perfiles de usuario $\mathbf{p}_u = \sum_{j \in H_u} w_j \mathbf{e}_j$ (pesos $w_j$ crecientes con la recencia) puntuamos $s(u,i) = \mathbf{p}_u^\top \mathbf{e}_i$.

**Diferencia clave con los embeddings de ID** (módulos 05/08): un embedding de ID se *aprende* de las interacciones → captura el gusto colectivo pero no existe para ítems nuevos. Un embedding de texto existe **desde el día 0** pero solo sabe lo que dice el texto.

| | Embedding de ID (CF) | Embedding de texto (LLM) |
|---|---|---|
| Ítem nuevo (cold start) | ❌ no existe | ✅ inmediato |
| Captura "gusto colectivo" | ✅ | ❌ (solo semántica) |
| Coste | tabla $\lvert\mathcal{I}\rvert\times d$ | inferencia del encoder |
| Multilingüe / consultas en lenguaje natural | ❌ | ✅ |

Usamos por defecto `paraphrase-multilingual-MiniLM-L12-v2` (rápido, multilingüe: permite consultas en español sobre títulos en inglés). Cambia `EMB_MODEL` a `BAAI/bge-m3` o `Qwen/Qwen3-Embedding-0.6B` para más calidad.

In [ ]:
def item_text(row) -> str:
    """Texto de ítem para el encoder: título limpio + año + géneros."""
    title = re.sub(r"\s*\(\d{4}\)\s*$", "", row.title)
    year = "" if pd.isna(row.year) else f" ({int(row.year)})"
    return f"{title}{year}. Géneros: {', '.join(row.genre_list)}."

class TextEncoder:
    """Encoder de texto: sentence-transformers si está disponible; si no, TF-IDF+SVD (LSA) de respaldo."""
    def __init__(self, model_name: str = EMB_MODEL):
        self.kind, self.model = "st", None
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_name, device=device)
        except Exception as e:
            print(f"⚠️ sentence-transformers/modelo no disponible ({type(e).__name__}): uso LSA de respaldo.")
            self.kind = "lsa"

    def fit(self, corpus):
        if self.kind == "lsa":
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(corpus)
            Xt = self.tfidf.transform(corpus)
            self.svd = TruncatedSVD(n_components=min(128, Xt.shape[1] - 1), random_state=SEED).fit(Xt)
        return self

    def encode(self, texts, batch_size: int = 128) -> np.ndarray:
        texts = list(texts)
        if self.kind == "st":
            E = self.model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                                  show_progress_bar=False)
        else:
            E = self.svd.transform(self.tfidf.transform(texts))
            E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
        return np.asarray(E, dtype=np.float32)

In [ ]:
movies["text"] = movies.apply(item_text, axis=1)
encoder = TextEncoder().fit(movies.text.tolist())
t0 = time.perf_counter()
E_text = encoder.encode(movies.text)
print(f"Embeddings de texto: {E_text.shape} en {time.perf_counter() - t0:.1f}s · encoder={encoder.kind}")
print("Ejemplo:", movies.text.iat[0])

In [ ]:
cf = PureSVDRecommender(X, dim=64)
text_retr = EmbeddingRetriever(E_text)
all_test_users = sorted(truth)

def content_profile(u, n_last=30):
    ids = hist_by_user[u][-n_last:]
    w = np.linspace(.5, 1., len(ids))[:, None]                    # más peso a lo reciente
    v = (w * E_text[ids]).sum(0); return v / (np.linalg.norm(v) + 1e-9)

def recommend_content(users, k=10):
    q = np.stack([content_profile(u) for u in users])
    res = text_retr.search(q, k, [set(hist_by_user[u]) for u in users])
    return {u: [i for i, _ in r] for u, r in zip(users, res)}

def recommend_pop(users, k=10):
    order = np.argsort(-pop)
    return {u: [int(i) for i in order if i not in set(hist_by_user[u])][:k] for u in users}

def recommend_hybrid(users, k=10, alpha=.7):
    """Fusión de puntuaciones z-normalizadas: alpha·CF + (1-alpha)·texto."""
    S_cf = cf.user_emb(users) @ cf.item_emb.T
    S_tx = np.stack([content_profile(u) for u in users]) @ E_text.T
    z = lambda S: (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
    S = alpha * z(S_cf) + (1 - alpha) * z(S_tx)
    out = {}
    for row, u in enumerate(users):
        S[row, hist_by_user[u]] = -np.inf
        out[u] = list(np.argsort(-S[row])[:k])
    return out

systems = {"Popularidad": recommend_pop, "Contenido (texto)": recommend_content,
           "CF (PureSVD)": lambda us, k: cf.recommend(us, k), "Híbrido CF+texto": recommend_hybrid}
res_enc = pd.DataFrame({name: evaluate(f(all_test_users, K), truth, K) for name, f in systems.items()}).T
res_enc

In [ ]:
ax = res_enc.plot.bar(figsize=(8, 3.6), rot=0, color=["#4c78a8", "#f58518"])
ax.set_title("Encoder de texto vs CF en MovieLens (split temporal)"); ax.set_ylabel("métrica @10")
for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=7)
plt.tight_layout(); plt.show()

🧪 **Lo esperable:** en usuarios e ítems *warm*, el CF gana claramente al contenido puro — el texto de MovieLens (título + géneros) es pobre. El híbrido suele mejorar algo al CF. Esto es lo que encuentran también los benchmarks serios: los LLMs aportan **sobre todo donde faltan interacciones**. Veámoslo en el espacio de embeddings y por cubos de popularidad.

In [ ]:
from sklearn.manifold import TSNE
idx = rng.choice(n_items, size=min(1000 if FAST_DEV_RUN else 3000, n_items), replace=False)
main_g = movies.genre_list.str[0].iloc[idx]
top_g = main_g.value_counts().index[:8]
Z = TSNE(n_components=2, random_state=SEED, perplexity=30, init="pca").fit_transform(E_text[idx])
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for g in top_g:
    m = (main_g == g).values
    ax.scatter(Z[m, 0], Z[m, 1], s=8, alpha=.7, label=g)
ax.set_title("t-SNE de embeddings de texto de películas (color = género principal)")
ax.legend(markerscale=2, fontsize=8, ncol=2); ax.set_xticks([]); ax.set_yticks([]); plt.show()

In [ ]:
# Hit-rate@50 por cubo de popularidad del ítem de test (en train)
recs_cf50 = cf.recommend(all_test_users, 50)
recs_tx50 = recommend_content(all_test_users, 50)
bins, labels = [-1, 0, 5, 20, 1e9], ["0 (cold)", "1-5", "6-20", ">20"]
rows = []
for u in all_test_users:
    s_cf, s_tx = set(recs_cf50[u]), set(recs_tx50[u])
    for i in truth[u]:
        rows.append((pd.cut([pop[i]], bins, labels=labels)[0], i in s_cf, i in s_tx))
hb = pd.DataFrame(rows, columns=["cubo", "CF", "Texto"]).groupby("cubo", observed=False)[["CF", "Texto"]].mean()
ax = hb.plot.bar(figsize=(7, 3.4), rot=0, color=["#4c78a8", "#54a24b"])
ax.set_title("Hit-rate@50 por popularidad del ítem en train"); ax.set_xlabel("nº interacciones del ítem en train")
plt.tight_layout(); plt.show(); hb

El CF **no puede** recuperar ítems con 0 interacciones (su embedding es el vector nulo), mientras que el encoder de texto sí. Ese es el argumento económico más sólido para usar LLMs como encoders: **cold start de ítems** (estrenos, catálogo de long tail, nuevos mercados). En §5 lo llevamos más lejos aprendiendo un mapeo texto → espacio CF.

## 3 · El LLM como ranker zero-shot (listwise)

### 📐 Formulación (Hou et al., 2024)
Dado el historial ordenado $H_u = (i_1, \dots, i_n)$ y un conjunto de candidatos $C = \{c_1, \dots, c_m\}$ **que produce un retriever**, el LLM devuelve una permutación $\pi$ de $C$:
$$\pi = \operatorname{LLM}(\operatorname{prompt}(H_u, C)).$$

Tres formas de preguntar:
| Estilo | Llamadas por usuario | Ventaja | Problema |
|---|---|---|---|
| **Pointwise** ("¿le gustará X?") | $m$ | calibrable (P(Yes)) | caro, no compara |
| **Pairwise** ("¿A o B?") | $O(m^2)$ o $O(m \log m)$ | robusto | carísimo |
| **Listwise** ("ordena estos $m$") | 1 | barato | **sesgo de posición**, omisiones, alucinaciones |

Hou et al. encontraron que los LLMs tienen capacidad zero-shot real, pero (1) **les cuesta percibir el orden** del historial, y (2) **están sesgados por la posición** de los candidatos en el prompt y por la **popularidad**. Remedios: prompts que enfatizan lo reciente y **bootstrapping**: barajar los candidatos $B$ veces y agregar con **Borda**:
$$\operatorname{score}(c) = \sum_{b=1}^{B} \big(m - \operatorname{rank}_b(c)\big).$$

Para listas largas, **RankGPT** (Sun et al., 2023) usa una ventana deslizante desde el final hacia el principio.

In [ ]:
# Precios USD por millón de tokens (entrada, salida). Verifícalos en la web del proveedor antes de presupuestar.
PRICES_PER_MTOK = {"claude-sonnet-5-5": (2.0, 10.0), "gpt-4o-mini": (0.15, 0.60)}

def approx_tokens(s: str) -> int:
    return max(1, len(s) // 4)          # regla rápida: ~4 caracteres por token en inglés

class BaseLLM:
    """Interfaz común. Todos los backends registran tokens y latencia de cada llamada."""
    model_name = "base"

    def __init__(self):
        self.log = []

    def generate(self, prompt: str, system: str = "", max_tokens: int = 512,
                 task: str = "generic", meta: dict | None = None) -> str:
        t0 = time.perf_counter()
        text, n_in, n_out = self._generate(prompt, system, max_tokens, task, meta or {})
        self.log.append({"task": task, "in_tok": n_in, "out_tok": n_out,
                         "latency_s": time.perf_counter() - t0})
        return text

    def yes_prob(self, prompt: str, system: str = "", meta: dict | None = None) -> float:
        """Probabilidad de 'sí' (scoring pointwise). Las APIs no dan logits → pedimos un número 0-100."""
        out = self.generate(prompt + "\nResponde SOLO con un número de 0 a 100: la probabilidad de que le guste.",
                            system, max_tokens=16, task="yesno", meta=meta)
        m = re.search(r"\d+(?:\.\d+)?", out)
        return min(100.0, float(m.group())) / 100 if m else 0.5

    def usage(self) -> pd.DataFrame:
        return pd.DataFrame(self.log, columns=["task", "in_tok", "out_tok", "latency_s"])

    def cost_usd(self) -> float:
        p_in, p_out = PRICES_PER_MTOK.get(self.model_name, (0.0, 0.0))
        df = self.usage()
        return float(df.in_tok.sum() * p_in + df.out_tok.sum() * p_out) / 1e6

In [ ]:
class MockLLM(BaseLLM):
    """LLM SIMULADO para ejecutar el notebook sin GPU ni API.

    ⚠️ No es un LLM: recibe `meta` (datos estructurados) y fabrica una salida con el MISMO formato
    que daría un LLM real (texto/JSON), incluyendo errores realistas (sesgo de posición, ítems
    omitidos o inventados). Sirve para probar tuberías, parsers y métricas. Las cifras obtenidas
    con el mock NO son resultados de LLMs: para eso ejecuta con LLM_BACKEND="hf"/"anthropic".
    """
    model_name = "mock"

    def __init__(self, ctx: dict, position_bias: float = 0.15, seed: int = SEED):
        super().__init__()
        self.ctx, self.position_bias = ctx, position_bias
        self.rng = np.random.default_rng(seed)

    def _profile(self, ids):
        return self.ctx["E"][list(ids)].mean(0) if len(ids) else np.zeros(self.ctx["E"].shape[1])

    def _generate(self, prompt, system, max_tokens, task, meta):
        E, pop, rng = self.ctx["E"], self.ctx["pop"], self.rng
        if task == "rank":
            cand = np.array(meta["cand_ids"]); n = len(cand)
            popn = np.log1p(pop[cand]) / np.log1p(pop.max())
            s = E[cand] @ self._profile(meta["hist_ids"]) + 0.3 * popn
            s = s + self.position_bias * (1 - np.arange(n) / n) + rng.normal(0, .05, n)
            order = list(np.argsort(-s) + 1)
            if rng.random() < .2: order = order[:-2]                 # omite ítems
            if rng.random() < .05: order.insert(3, n + 3)            # inventa un número
            out = ", ".join(map(str, order))
        elif task == "yesno":
            h = self._profile(meta["like_ids"]) - 0.5 * self._profile(meta["dislike_ids"])
            p = 1 / (1 + np.exp(-(6 * float(E[meta["target_id"]] @ h) + rng.normal(0, .5) - 1)))
            out = f"{100 * p:.0f}"
        elif task == "enrich":
            g = self.ctx["genres"][meta["item_id"]]
            out = json.dumps({"sinopsis": f"Una historia de {', '.join(g).lower()}.", "temas": g[:3],
                              "tono": "oscuro" if set(g) & {"Horror", "Thriller", "Crime"} else "ligero",
                              "publico": "familiar" if "Children" in g else "adulto",
                              "palabras_clave": [x.lower() for x in g]}, ensure_ascii=False)
        elif task == "judge":
            prof = self._profile(meta["hist_ids"])
            sa, sb = (float(E[meta[k]].mean(0) @ prof) for k in ("list_a", "list_b"))
            w = "A" if rng.random() < .15 else ("A" if sa > sb + .01 else "B" if sb > sa + .01 else "empate")
            out = json.dumps({"ganador": w, "confianza": 3, "motivo": "afinidad de géneros (mock)"})
        elif task == "explain":
            ev = meta["evidence_titles"]
            if rng.random() < .15:                                    # alucinación simulada
                ev = ev[:1] + [self.ctx["titles"][int(rng.integers(len(self.ctx["titles"])))]]
            out = (f"Te la recomiendo porque disfrutaste \"{ev[0]}\"" +
                   (f" y \"{ev[1]}\"" if len(ev) > 1 else "") + ", con un estilo parecido.")
        elif task == "simulate":
            taste = meta["taste"]
            dec = [{"id": int(i), "ver": bool(rng.random() < 1 / (1 + np.exp(-(8 * float(E[i] @ taste) - 2))))}
                   for i in meta["page_ids"]]
            out = json.dumps(dec)
        elif task == "hyde":
            out = meta.get("query", prompt)
        else:
            out = "OK (respuesta simulada por MockLLM)"
        return out, approx_tokens(system + prompt), approx_tokens(out)

In [ ]:
class HFLLM(BaseLLM):
    """Modelo abierto con 🤗 transformers. 7-8B: bf16 en L4/A100; en T4 (16 GB) se carga en 4-bit (NF4)."""
    def __init__(self, model_name: str = HF_MODEL, load_in_4bit: bool | None = None):
        super().__init__()
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        self.model_name = model_name
        self.tok = AutoTokenizer.from_pretrained(model_name)
        if load_in_4bit is None:
            load_in_4bit = torch.cuda.get_device_properties(0).total_memory < 20e9
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        kw = {"device_map": "auto", "torch_dtype": dtype}
        if load_in_4bit:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=dtype)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **kw).eval()

    def _chat_text(self, prompt, system):
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        extra = {"enable_thinking": False} if "Qwen3" in self.model_name else {}   # Qwen3: sin "thinking"
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **extra)

    @torch.no_grad() if torch is not None else (lambda f: f)
    def _generate(self, prompt, system, max_tokens, task, meta):
        enc = self.tok(self._chat_text(prompt, system), return_tensors="pt").to(self.model.device)
        out = self.model.generate(**enc, max_new_tokens=max_tokens, do_sample=False,
                                  pad_token_id=self.tok.eos_token_id)
        new = out[0, enc.input_ids.shape[1]:]
        return self.tok.decode(new, skip_special_tokens=True), enc.input_ids.shape[1], len(new)

    @torch.no_grad() if torch is not None else (lambda f: f)
    def yes_prob(self, prompt, system="", meta=None):
        """P(Yes) leyendo los logits del siguiente token (como TALLRec): sin generar texto."""
        t0 = time.perf_counter()
        enc = self.tok(self._chat_text(prompt + "\nAnswer only Yes or No.", system),
                       return_tensors="pt").to(self.model.device)
        logits = self.model(**enc).logits[0, -1]
        yes = self.tok.encode("Yes", add_special_tokens=False)[0]
        no = self.tok.encode("No", add_special_tokens=False)[0]
        p = torch.softmax(logits[[yes, no]].float(), 0)[0].item()
        self.log.append({"task": "yesno", "in_tok": enc.input_ids.shape[1], "out_tok": 0,
                         "latency_s": time.perf_counter() - t0})
        return p

class VLLMLLM(BaseLLM):
    """Mismo modelo servido con vLLM (PagedAttention + continuous batching): x5-x20 throughput en lotes."""
    def __init__(self, model_name: str = HF_MODEL):
        super().__init__()
        from vllm import LLM, SamplingParams
        self.model_name, self.SamplingParams = model_name, SamplingParams
        self.llm = LLM(model=model_name, max_model_len=4096, gpu_memory_utilization=0.85)

    def _generate(self, prompt, system, max_tokens, task, meta):
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        o = self.llm.chat([msgs], self.SamplingParams(temperature=0, max_tokens=max_tokens), use_tqdm=False)[0]
        return o.outputs[0].text, len(o.prompt_token_ids), len(o.outputs[0].token_ids)

In [ ]:
class AnthropicLLM(BaseLLM):
    """Claude vía API (pip install anthropic; variable ANTHROPIC_API_KEY)."""
    def __init__(self, model_name: str = ANTHROPIC_MODEL, effort: str = "low"):
        super().__init__()
        import anthropic
        self.client, self.model_name, self.effort = anthropic.Anthropic(), model_name, effort

    def _generate(self, prompt, system, max_tokens, task, meta):
        kw = dict(model=self.model_name, max_tokens=max(max_tokens, 2048),   # margen para el razonamiento
                  messages=[{"role": "user", "content": prompt}],
                  output_config={"effort": self.effort})                   # tareas cortas → esfuerzo bajo
        if system:
            kw["system"] = system
        try:   # fallback de servidor ante rechazos (beta); si tu SDK no lo soporta, llamada estándar
            resp = self.client.beta.messages.create(**kw, betas=["server-side-fallback-2026-07-01"],
                                                    fallbacks="default")
        except TypeError:
            resp = self.client.messages.create(**kw)
        if resp.stop_reason == "refusal":
            return "", resp.usage.input_tokens, resp.usage.output_tokens
        text = "".join(b.text for b in resp.content if b.type == "text")
        return text, resp.usage.input_tokens, resp.usage.output_tokens

class OpenAILLM(BaseLLM):
    """Modelos de OpenAI (pip install openai; variable OPENAI_API_KEY)."""
    def __init__(self, model_name: str = OPENAI_MODEL):
        super().__init__()
        from openai import OpenAI
        self.client, self.model_name = OpenAI(), model_name

    def _generate(self, prompt, system, max_tokens, task, meta):
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        r = self.client.chat.completions.create(model=self.model_name, messages=msgs,
                                                temperature=0, max_tokens=max_tokens)
        return r.choices[0].message.content or "", r.usage.prompt_tokens, r.usage.completion_tokens

def make_llm(backend: str = LLM_BACKEND, ctx: dict | None = None) -> BaseLLM:
    """'auto': GPU → modelo abierto (hf); si no, API (Anthropic/OpenAI) si hay clave; si no, mock."""
    if backend == "auto":
        backend = ("hf" if device == "cuda" else "anthropic" if os.environ.get("ANTHROPIC_API_KEY")
                   else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
    builders = {"hf": HFLLM, "vllm": VLLMLLM, "anthropic": AnthropicLLM, "openai": OpenAILLM}
    if backend in builders:
        try:
            llm = builders[backend]()
            print(f"✅ LLM listo: {backend} · {llm.model_name}")
            return llm
        except Exception as e:
            print(f"⚠️ No se pudo crear el backend '{backend}' ({type(e).__name__}: {e}). Uso MockLLM.")
    print("ℹ️ Backend MockLLM: el pipeline se ejecuta entero, pero las cifras NO son de un LLM real.")
    return MockLLM(ctx)

In [ ]:
ctx = {"E": E_text, "pop": pop, "genres": movies.genre_list.tolist(), "titles": movies.title.tolist()}
llm = make_llm(LLM_BACKEND, ctx)

In [ ]:
SYSTEM_RANKER = "Eres un sistema de recomendación de películas. Respondes únicamente con el formato pedido."

def short_title(i: int) -> str:
    return movies.title.iat[i]

def build_listwise_prompt(hist_ids, cand_ids) -> str:
    hist = "\n".join(f"- {short_title(i)}" for i in hist_ids)
    cands = "\n".join(f"[{j + 1}] {short_title(i)}" for j, i in enumerate(cand_ids))
    return (f"Historial del usuario en orden cronológico (la última es la más reciente):\n{hist}\n\n"
            f"Ordena las {len(cand_ids)} películas candidatas de MÁS a MENOS probable que quiera ver ahora:\n"
            f"{cands}\n\nResponde SOLO con los números entre corchetes separados por comas, "
            f"sin texto adicional. Ejemplo: 3, 1, 2")

def parse_ranking(text: str, n: int):
    """Parser robusto: ignora años (4 dígitos), duplicados e índices inventados; completa los omitidos."""
    nums = [int(x) for x in re.findall(r"(?<!\d)(\d{1,3})(?!\d)", text)]
    order, seen, invalid = [], set(), 0
    for x in nums:
        if 1 <= x <= n and x not in seen:
            order.append(x - 1); seen.add(x)
        elif not 1 <= x <= n:
            invalid += 1
    missing = [j for j in range(n) if j + 1 not in seen]
    return order + missing, {"invalid": invalid, "missing": len(missing)}

def llm_rerank(llm, hist_ids, cand_ids, n_perm: int = 1, rng=None, stats: list | None = None):
    """Re-ranking listwise; con n_perm>1 baraja los candidatos y agrega con Borda (bootstrapping)."""
    rng = rng or np.random.default_rng(SEED)
    n, borda = len(cand_ids), collections.Counter()
    for b in range(n_perm):
        perm = list(cand_ids) if b == 0 else list(rng.permutation(cand_ids))
        txt = llm.generate(build_listwise_prompt(hist_ids, perm), SYSTEM_RANKER, max_tokens=8 * n,
                           task="rank", meta={"cand_ids": perm, "hist_ids": list(hist_ids)})
        order, st = parse_ranking(txt, n)
        if stats is not None:
            stats.append(st)
        for r, j in enumerate(order):
            borda[perm[j]] += n - r
    return [i for i, _ in borda.most_common()]

In [ ]:
u0 = eval_users[0]
cands0 = cf.recommend([u0], 20)[u0]
p = build_listwise_prompt(hist_by_user[u0][-15:], cands0)
print(p[:1500], "\n...")
out = llm.generate(p, SYSTEM_RANKER, max_tokens=160, task="rank",
                   meta={"cand_ids": cands0, "hist_ids": hist_by_user[u0][-15:]})
print("\nRespuesta cruda del LLM:", out[:300])
print("Parseada:", parse_ranking(out, len(cands0)))

### 🧪 Experimento 1: ¿mejora el LLM el orden del retriever?

Protocolo **realista**: el LLM reordena el top-20 *real* del CF. Si el ítem futuro no está en el top-20, ningún re-ranker puede recuperarlo (techo = Recall@20 del retriever). Comparamos NDCG@10 del orden del CF vs. el orden del LLM (1 pasada) vs. LLM con bootstrapping $B=3$.

> ⚠️ Muchos papers usan el protocolo "1 positivo + 19 negativos aleatorios": los negativos aleatorios son fáciles y **inflan** los resultados. Aquí los candidatos son los difíciles: los que el CF ya considera buenos.

In [ ]:
N_CANDS, N_HIST = 20, 15
cf_top = cf.recommend(eval_users, N_CANDS)
rank_stats, rows = [], []
for u in eval_users:
    hist, cands = hist_by_user[u][-N_HIST:], cf_top[u]
    r1 = llm_rerank(llm, hist, cands, n_perm=1, stats=rank_stats)
    r3 = llm_rerank(llm, hist, cands, n_perm=3, rng=np.random.default_rng(u))
    rows.append({"user": u, "techo (Recall@20)": float(len(set(cands) & truth[u]) > 0),
                 "CF": ndcg_at_k(cands, truth[u]), "LLM listwise": ndcg_at_k(r1, truth[u]),
                 "LLM + bootstrap B=3": ndcg_at_k(r3, truth[u])})
res_rank = pd.DataFrame(rows)
res_rank.drop(columns="user").mean().to_frame("media")

In [ ]:
m = res_rank.drop(columns=["user", "techo (Recall@20)"])
mean, se = m.mean(), m.std() / np.sqrt(len(m))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(mean.index, mean.values, yerr=1.96 * se.values, capsize=4, color=["#4c78a8", "#e45756", "#f58518"])
ax[0].set_title(f"NDCG@10 re-ranking del top-{N_CANDS} (n={len(m)} usuarios, IC 95%)"); ax[0].tick_params(axis="x", labelsize=8)
st = pd.DataFrame(rank_stats)
ax[1].hist([st.invalid, st.missing], bins=range(0, 8), label=["índices inventados", "candidatos omitidos"], color=["#e45756", "#bab0ac"])
ax[1].set_title("Errores de formato del LLM por llamada"); ax[1].legend(); ax[1].set_xlabel("nº por respuesta")
plt.tight_layout(); plt.show()
print(f"Llamadas con alguna alucinación de índice: {(st.invalid > 0).mean():.1%} · con omisiones: {(st.missing > 0).mean():.1%}")

Fíjate en el **intervalo de confianza**: con 40 usuarios (FAST_DEV_RUN) las diferencias entre métodos suelen no ser significativas. Antes de concluir nada, sube `N_EVAL_USERS` (módulo 02: bootstrap y significancia). Y fíjate en el panel derecho: **siempre** hay que parsear defensivamente. En producción, un índice inventado o un ítem omitido no puede romper la página.

### 🧪 Experimento 2: sesgo de posición
Ponemos el ítem verdadero en distintas posiciones de la lista de candidatos (el resto, barajado) y medimos en qué puesto lo devuelve el LLM. Un ranker sin sesgo daría una línea plana.

In [ ]:
M_POS = 8 if FAST_DEV_RUN else 40
positions = [0, 5, 10, 15, 19]
pos_rows = []
users_pos = [u for u in eval_users if truth[u]][:M_POS]
for u in users_pos:
    target = next(iter(truth[u]))
    negs = [i for i in cf_top[u] if i != target][:N_CANDS - 1]
    for p_in in positions:
        cands = list(np.random.default_rng(u + p_in).permutation(negs)); cands.insert(p_in, target)
        out = llm_rerank(llm, hist_by_user[u][-N_HIST:], cands, n_perm=1)
        pos_rows.append({"pos_entrada": p_in, "rank_salida": out.index(target)})
pb = pd.DataFrame(pos_rows).groupby("pos_entrada").rank_salida.agg(["mean", "sem"])
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.errorbar(pb.index, pb["mean"], yerr=1.96 * pb["sem"], marker="o", capsize=4)
ax.axhline((N_CANDS - 1) / 2, ls="--", c="gray", label="sin información")
ax.set_xlabel("posición del ítem verdadero en el prompt"); ax.set_ylabel("puesto medio en la salida")
ax.set_title("Sesgo de posición del ranker LLM (menor = mejor)"); ax.legend(); plt.show()

In [ ]:
# Bootstrapping: calidad vs coste (nº de llamadas)
users_b = eval_users[: (10 if FAST_DEV_RUN else 60)]
curve = []
for B in [1, 3, 5]:
    nd = [ndcg_at_k(llm_rerank(llm, hist_by_user[u][-N_HIST:], cf_top[u], n_perm=B,
                               rng=np.random.default_rng(u)), truth[u]) for u in users_b]
    curve.append({"B": B, "NDCG@10": np.mean(nd), "llamadas/usuario": B})
curve = pd.DataFrame(curve)
fig, ax = plt.subplots(figsize=(6, 3.2)); ax2 = ax.twinx()
ax.plot(curve.B, curve["NDCG@10"], "o-", c="#e45756", label="NDCG@10"); ax.set_ylabel("NDCG@10", color="#e45756")
ax2.bar(curve.B, curve["llamadas/usuario"], alpha=.2, width=.6); ax2.set_ylabel("llamadas al LLM por usuario")
ax.set_xlabel("permutaciones B (Borda)"); ax.set_title("Bootstrapping: robustez a cambio de coste lineal"); plt.show()
curve

## 4 · Fine-tuning con LoRA al estilo TALLRec

### 💡 Por qué
TALLRec (Bao et al., RecSys 2023) observó que LLMs genéricos usados con *in-context learning* rinden cerca del azar (AUC ≈ 0,5) en predecir si a un usuario le gustará una película o un libro, pero que **un ajuste ligero con LoRA y muy pocos ejemplos** (desde decenas a cientos) los vuelve competitivos y con buena generalización entre dominios. La tarea se plantea como clasificación binaria en lenguaje natural:

```
### Instruction: Given the user's preference and unpreference, identify whether the user will like the target movie by answering "Yes." or "No.".
### Input: User Preference: "A", "B", ...  User Unpreference: "C", ...  Whether the user will like the target movie "X"?
### Response: Yes.
```
y se evalúa con **AUC** usando $P(\text{Yes})$ del primer token generado.

### 📐 LoRA en una ecuación
En lugar de actualizar una matriz $W_0 \in \mathbb{R}^{d \times k}$ (congelada), aprendemos una actualización de **rango bajo**:
$$W = W_0 + \frac{\alpha}{r} B A, \qquad B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times k},\ r \ll \min(d,k)$$
- $B$ se inicializa a cero → al principio $W = W_0$ (el modelo no cambia).
- Parámetros entrenables: $r(d+k)$ en vez de $dk$. Con $d=k=4096$ y $r=8$: 65.536 frente a 16,8 M (**0,4 %**).
- **QLoRA** (Dettmers et al., 2023): $W_0$ en 4-bit NF4 → un 7B cabe en ~5–6 GB.

Antes de usar `peft`, veamos el mecanismo desde cero en NumPy: aprender una actualización de rango bajo con descenso de gradiente.

In [ ]:
def lora_from_scratch(r, d=64, k=64, true_rank=4, n=500, steps=300, lr=0.1, seed=SEED):
    """W* = W0 + U V^T (rango true_rank). Congelamos W0 y aprendemos B (d×r) y A (r×k)."""
    g = np.random.default_rng(seed)
    W0 = g.normal(0, 1 / np.sqrt(k), (d, k))
    W_star = W0 + g.normal(0, .15, (d, true_rank)) @ g.normal(0, .15, (true_rank, k))
    Xin = g.normal(0, 1, (n, k)); Y = Xin @ W_star.T
    A = g.normal(0, 1 / np.sqrt(k), (r, k)); B = np.zeros((d, r))      # B=0 → arranca en W0
    losses = []
    for _ in range(steps):
        pred = Xin @ (W0 + B @ A).T
        gW = (2 * (pred - Y) / n).T @ Xin                                # dL/dW (d×k)
        B, A = B - lr * gW @ A.T, A - lr * B.T @ gW                      # regla de la cadena: dL/dB, dL/dA
        losses.append(float(((pred - Y) ** 2).mean()))
    return losses, r * (d + k)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
finals = {}
for r in [1, 2, 4, 8, 16]:
    l, n_par = lora_from_scratch(r); finals[r] = (l[-1], n_par)
    ax[0].plot(l, label=f"r={r}")
ax[0].set_yscale("log"); ax[0].set_title("LoRA desde cero: pérdida (rango real = 4)"); ax[0].set_xlabel("paso"); ax[0].legend()
rs = list(finals)
ax[1].bar([str(r) for r in rs], [finals[r][1] for r in rs], color="#72b7b2")
ax[1].axhline(64 * 64, c="r", ls="--", label="fine-tuning completo (d·k)")
ax[1].set_title("Parámetros entrenables"); ax[1].set_xlabel("rango r"); ax[1].legend()
plt.tight_layout(); plt.show()

Con $r <$ rango real la pérdida se estanca (no hay capacidad); con $r \geq 4$ se ajusta con una fracción de los parámetros. En LLMs la hipótesis (Hu et al., 2021) es que la **adaptación a una tarea tiene rango intrínseco bajo**.

### Dataset TALLRec desde MovieLens
Para cada usuario tomamos objetivos en su secuencia temporal; el historial son las 10 interacciones previas separadas en *preference* (rating > 3) y *unpreference*; la etiqueta es si el objetivo tiene rating > 3. Separamos **por usuario** (train/valid/test) para no filtrar información.

In [ ]:
def tallrec_prompt(like_titles, dislike_titles, target_title) -> str:
    q = lambda ts: ", ".join('"' + t + '"' for t in ts) or "None"
    return ("### Instruction:\nGiven the user's preference and unpreference, identify whether the user will "
            "like the target movie by answering \"Yes.\" or \"No.\".\n\n### Input:\n"
            f"User Preference: {q(like_titles)}\nUser Unpreference: {q(dislike_titles)}\n"
            f"Whether the user will like the target movie \"{target_title}\"?\n\n### Response:")

def build_tallrec(ratings, n_hist=10, per_user=4, seed=SEED):
    g, rows = np.random.default_rng(seed), []
    for u, grp in ratings.groupby("user_id"):
        its, rts = grp.item_id.to_numpy(), grp.rating.to_numpy()
        if len(its) <= n_hist:
            continue
        for t in g.choice(np.arange(n_hist, len(its)), size=min(per_user, len(its) - n_hist), replace=False):
            h, hr = its[t - n_hist:t], rts[t - n_hist:t]
            like, dislike = list(h[hr > 3]), list(h[hr <= 3])
            rows.append({"user_id": u, "target": int(its[t]), "label": int(rts[t] > 3),
                         "like_ids": like, "dislike_ids": dislike,
                         "prompt": tallrec_prompt([short_title(i) for i in like],
                                                  [short_title(i) for i in dislike], short_title(its[t]))})
    df = pd.DataFrame(rows)
    users = df.user_id.unique(); g.shuffle(users)
    cut1, cut2 = int(.8 * len(users)), int(.9 * len(users))
    df["split"] = df.user_id.map({**{u: "train" for u in users[:cut1]}, **{u: "valid" for u in users[cut1:cut2]},
                                  **{u: "test" for u in users[cut2:]}})
    return df

tall = build_tallrec(ratings)
print(tall.groupby("split").label.agg(["size", "mean"]).rename(columns={"mean": "% Yes"}))
print("\n" + tall.prompt.iat[0])

### Baselines antes que LLMs
🧠 Un principio de la élite: **nunca** publiques un resultado de LLM-ranker sin un baseline clásico al lado. Aquí, una regresión logística con tres *features* obvias: rating medio del usuario, rating medio del ítem (en train) y afinidad de contenido (coseno entre el objetivo y el perfil de *likes* menos *dislikes*).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

tr_users = set(tall.query("split=='train'").user_id)
r_tr = ratings[ratings.user_id.isin(tr_users)]
item_mean = r_tr.groupby("item_id").rating.mean().reindex(range(n_items)).fillna(r_tr.rating.mean()).to_numpy()

def feats(df):
    um = [np.mean([rating_lookup[(u, i)] for i in l + d]) for u, l, d in zip(df.user_id, df.like_ids, df.dislike_ids)]
    def aff(l, d, t):
        p = (E_text[l].mean(0) if l else 0) - (E_text[d].mean(0) if d else 0)
        return float(E_text[t] @ p) if np.ndim(p) else 0.0
    af = [aff(l, d, t) for l, d, t in zip(df.like_ids, df.dislike_ids, df.target)]
    return np.c_[um, item_mean[df.target], af]

tr, te = tall.query("split=='train'"), tall.query("split=='test'")
lr_clf = LogisticRegression(max_iter=1000).fit(feats(tr), tr.label)
auc_results = {"Azar": 0.5, "LogReg (3 features)": roc_auc_score(te.label, lr_clf.predict_proba(feats(te))[:, 1])}

N_ZS = 60 if FAST_DEV_RUN else 400
te_zs = te.sample(min(N_ZS, len(te)), random_state=SEED)
p_zs = [llm.yes_prob(p, meta={"like_ids": l, "dislike_ids": d, "target_id": t})
        for p, l, d, t in zip(te_zs.prompt, te_zs.like_ids, te_zs.dislike_ids, te_zs.target)]
auc_results[f"LLM zero-shot ({llm.model_name})"] = roc_auc_score(te_zs.label, p_zs)
pd.Series(auc_results, name="AUC").round(3)

### Entrenamiento LoRA con `peft` (GPU)
- `LORA_BASE`: en `FAST_DEV_RUN` un **Qwen2.5-1.5B-Instruct** (cabe en T4 en fp32 y entrena en minutos). Con `FAST_DEV_RUN=False` usa `HF_MODEL` (7–8B) en **QLoRA** 4-bit: en L4 ~30–60 min para unos miles de ejemplos.
- Hiperparámetros típicos de TALLRec: $r=8$, $\alpha=16$, dropout 0,05, módulos `q_proj`/`v_proj`, lr $\sim 10^{-4}$.
- La pérdida solo se calcula sobre la respuesta (*labels* = −100 en el prompt).

> ⚠️ Si cargaste un 7B con el backend `hf` en una T4, libera memoria antes (`del llm; torch.cuda.empty_cache()`) o usa el modelo pequeño.

In [ ]:
RUN_LORA = device == "cuda"
LORA_BASE = "Qwen/Qwen2.5-1.5B-Instruct" if FAST_DEV_RUN else HF_MODEL
N_TRAIN_LORA = 512 if FAST_DEV_RUN else 4096
lora_losses = []
if RUN_LORA:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    use_4bit = any(s in LORA_BASE for s in ["7B", "8B"])
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    tok = AutoTokenizer.from_pretrained(LORA_BASE)
    tok.pad_token = tok.pad_token or tok.eos_token
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype) if use_4bit else None
    base = AutoModelForCausalLM.from_pretrained(LORA_BASE, device_map="auto", torch_dtype=dtype, quantization_config=qcfg)
    if use_4bit:
        base = prepare_model_for_kbit_training(base)
    model = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                                            target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    YES_ID = tok(" Yes", add_special_tokens=False).input_ids[0]
    NO_ID = tok(" No", add_special_tokens=False).input_ids[0]

    @torch.no_grad()
    def lora_yes_prob(prompts, bs=8):
        model.eval(); tok.padding_side = "left"; out = []
        for b in range(0, len(prompts), bs):
            enc = tok(list(prompts[b:b + bs]), return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            lg = model(**enc).logits[:, -1, [YES_ID, NO_ID]].float()
            out += torch.softmax(lg, -1)[:, 0].tolist()
        return np.array(out)

    te_l = te.sample(min(400, len(te)), random_state=SEED)
    auc_results["Base sin ajustar (P(Yes))"] = roc_auc_score(te_l.label, lora_yes_prob(te_l.prompt.tolist()))
else:
    print("Sin GPU: se omite el fine-tuning LoRA (el resto del notebook sigue funcionando).")

In [ ]:
if RUN_LORA:
    def encode_example(prompt, label, max_len=512):
        p = tok(prompt, add_special_tokens=False).input_ids
        a = tok(" Yes." if label else " No.", add_special_tokens=False).input_ids + [tok.eos_token_id]
        ids = (p + a)[-max_len:]; lab = ([-100] * len(p) + a)[-max_len:]
        return ids, lab

    tr_l = tr.sample(min(N_TRAIN_LORA, len(tr)), random_state=SEED)
    data = [encode_example(p, y) for p, y in zip(tr_l.prompt, tr_l.label)]
    opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=2e-4)
    BS, EPOCHS = 8, 2
    model.train()
    for ep in range(EPOCHS):
        order = np.random.default_rng(ep).permutation(len(data))
        for b in range(0, len(order), BS):
            batch = [data[j] for j in order[b:b + BS]]
            L = max(len(x[0]) for x in batch)
            ids = torch.tensor([x[0] + [tok.pad_token_id] * (L - len(x[0])) for x in batch], device=model.device)
            lab = torch.tensor([x[1] + [-100] * (L - len(x[1])) for x in batch], device=model.device)
            att = torch.tensor([[1] * len(x[0]) + [0] * (L - len(x[0])) for x in batch], device=model.device)
            loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            loss.backward(); opt.step(); opt.zero_grad()
            lora_losses.append(loss.item())
    auc_results[f"LoRA TALLRec (n={len(tr_l)})"] = roc_auc_score(te_l.label, lora_yes_prob(te_l.prompt.tolist()))

fig, ax = plt.subplots(1, 2 if lora_losses else 1, figsize=(11 if lora_losses else 6, 3.4), squeeze=False)
s = pd.Series(auc_results)
ax[0, 0].barh(s.index, s.values, color="#4c78a8"); ax[0, 0].axvline(.5, c="r", ls="--")
ax[0, 0].set_xlim(.4, max(.85, s.max() + .03)); ax[0, 0].set_title("AUC en usuarios de test (¿le gustará?)")
if lora_losses:
    ax[0, 1].plot(pd.Series(lora_losses).rolling(10, min_periods=1).mean()); ax[0, 1].set_title("Pérdida LoRA (media móvil)")
plt.tight_layout(); plt.show()

**Cómo leer el resultado.** Lo habitual: el LLM zero-shot queda cerca del azar o algo por encima; la regresión logística con 3 *features* es un rival duro; el LoRA mejora mucho sobre su base y se acerca o supera al baseline con pocos cientos de ejemplos. En el paper, la ventaja de TALLRec es mayor en el régimen **few-shot** y **entre dominios** (entrenar en películas y evaluar en libros).

🧪 **Experimento sugerido:** repite con `N_TRAIN_LORA` ∈ {64, 256, 1024, 4096} y dibuja AUC vs. nº de ejemplos para ambos modelos (LoRA y LogReg). Es la curva de eficiencia de muestras que vende TALLRec.

## 5 · Enriquecimiento de metadatos y cold start con LLMs

### 💡 La idea
El LLM conoce el mundo: sabe de qué va *Alien* aunque tu catálogo solo diga "Horror|Sci-Fi". **Offline**, una vez por ítem, le pedimos metadatos estructurados (sinopsis, temas, tono, público, palabras clave) y los usamos como features o para re-embeddings. Es el patrón de **KAR** (Xi et al., 2023, Huawei: conocimiento "razonado" sobre preferencias y "factual" sobre ítems, adaptado como features para el modelo CTR) y de **LLMRec** (Wei et al., WSDM 2024: aumentar el grafo con aristas, atributos y perfiles generados).

Reglas de producción:
1. **Salida estructurada** (JSON con esquema) + **validación**; lo que no valida se descarta o reintenta.
2. **Caché** por (ítem, versión del prompt, modelo): nunca pagues dos veces por lo mismo.
3. **Permitir "no lo sé"**: si el LLM no conoce un título, que lo diga en vez de inventar (y mide esa tasa).
4. **Batch**: vLLM offline o las APIs de *batch* (más baratas y sin límite de latencia).

In [ ]:
ENRICH_SYSTEM = ("Eres un documentalista de cine riguroso. Si no conoces la película con seguridad, "
                 "responde con \"sinopsis\": \"desconocida\". Nunca inventes datos.")
ENRICH_KEYS = {"sinopsis": str, "temas": list, "tono": str, "publico": str, "palabras_clave": list}

def enrich_prompt(i: int) -> str:
    return (f"Película: {movies.title.iat[i]}\nGéneros: {', '.join(movies.genre_list.iat[i])}\n"
            "Devuelve SOLO un objeto JSON con estas claves: \"sinopsis\" (máx. 25 palabras), \"temas\" (lista, máx. 4), "
            "\"tono\" (una palabra), \"publico\" (\"infantil\"|\"familiar\"|\"adulto\"), \"palabras_clave\" (lista, máx. 6).")

def parse_json(text: str):
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}|\[.*\]", text, flags=re.S)
        try:
            return json.loads(m.group()) if m else None
        except Exception:
            return None

def validate_enrichment(d) -> bool:
    return (isinstance(d, dict) and all(isinstance(d.get(k), t) for k, t in ENRICH_KEYS.items())
            and d["publico"] in {"infantil", "familiar", "adulto"})

def enrich_items(llm, item_ids, cache_path):
    cache = {}
    if Path(cache_path).exists():
        cache = {int(r["item_id"]): r for r in map(json.loads, open(cache_path))}
    with open(cache_path, "a") as f:
        for i in item_ids:
            if int(i) in cache:
                continue
            d = parse_json(llm.generate(enrich_prompt(i), ENRICH_SYSTEM, max_tokens=300, task="enrich",
                                        meta={"item_id": int(i)}))
            rec = {"item_id": int(i), "ok": validate_enrichment(d), **(d if validate_enrichment(d) else {})}
            cache[int(i)] = rec; f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return cache

In [ ]:
# Elegimos los ítems "cold" del experimento siguiente + algunos populares
g = np.random.default_rng(SEED)
eligible = np.where(pop >= 5)[0]
N_COLD = 60 if FAST_DEV_RUN else 500
cold_items = np.sort(g.choice(eligible, size=min(N_COLD, len(eligible)), replace=False))
enrich_ids = list(cold_items) + list(np.argsort(-pop)[:20])
cache_file = DATA_DIR / f"enrich_{llm.model_name.replace('/', '_')}.jsonl"
t0 = time.perf_counter(); n_calls0 = len(llm.log)
enriched = enrich_items(llm, enrich_ids, cache_file)
df_en = pd.DataFrame(enriched.values())
print(f"{len(df_en)} ítems · JSON válido: {df_en.ok.mean():.0%} · 'desconocida': "
      f"{(df_en.get('sinopsis') == 'desconocida').mean():.0%} · {len(llm.log) - n_calls0} llamadas en {time.perf_counter() - t0:.1f}s")
df_en.assign(title=df_en.item_id.map(movies.title)).head(5)

### 🧪 Cold start de ítems: mapear texto → espacio colaborativo
Simulamos estrenos: quitamos **todas** las interacciones de `cold_items` del train, reentrenamos el CF y comparamos cómo se recomiendan esos ítems a los usuarios que de verdad los vieron:

1. **Azar** dentro del pool cold.
2. **Texto directo**: perfil de contenido del usuario · embedding de texto del ítem.
3. **Texto → CF (mapeo lineal)**: aprendemos una regresión *ridge* $\hat{\mathbf{v}}_i = W^\top \mathbf{e}_i$ con los ítems *warm* (de embedding de texto a factor CF) y la aplicamos a los cold. Es la idea de DropoutNet/"content-to-CF mapping": hablar el idioma del CF sin interacciones.
4. **Texto enriquecido por el LLM** (sinopsis+temas+palabras clave añadidas al texto).
5. **Oráculo**: CF entrenado *con* sus interacciones (techo).

In [ ]:
from sklearn.linear_model import Ridge
cold_set = set(cold_items.tolist())
train_warm = train[~train.item_id.isin(cold_set)]
X_warm = build_interaction_matrix(train_warm, n_users, n_items)
cf_warm = PureSVDRecommender(X_warm, dim=64)
warm = np.setdiff1d(np.where(np.asarray(X_warm.sum(0)).ravel() > 0)[0], cold_items)
mapper = Ridge(alpha=1.0).fit(E_text[warm], cf_warm.item_emb[warm])

def enriched_text(i):
    d = enriched.get(int(i), {})
    extra = " ".join([d.get("sinopsis", "")] + d.get("temas", []) + d.get("palabras_clave", [])) if d.get("ok") else ""
    return movies.text.iat[i] + " " + extra
E_cold_en = encoder.encode([enriched_text(i) for i in cold_items])

truth_cold = train[train.item_id.isin(cold_set)].groupby("user_id").item_id.apply(set).to_dict()
users_c = [u for u in truth_cold if len(hist_by_user[u]) > len(truth_cold[u])][:500]
U_cf = cf_warm.user_emb(users_c); U_or = cf.user_emb(users_c)
prof = np.stack([E_text[[i for i in hist_by_user[u] if i not in cold_set][-30:]].mean(0) for u in users_c])
scorers = {"Azar": np.random.default_rng(1).random((len(users_c), len(cold_items))),
           "Texto directo": prof @ E_text[cold_items].T,
           "Texto enriquecido (LLM)": prof @ E_cold_en.T,
           "Texto → CF (ridge)": U_cf @ mapper.predict(E_text[cold_items]).T,
           "Oráculo CF (con datos)": U_or @ cf.item_emb[cold_items].T}
res_cold = {name: np.mean([recall_at_k(list(cold_items[np.argsort(-S[r])]), truth_cold[u], 10)
                           for r, u in enumerate(users_c)]) for name, S in scorers.items()}
s = pd.Series(res_cold)
ax = s.plot.barh(figsize=(7, 3), color=["#bab0ac", "#54a24b", "#9ecae9", "#f58518", "#4c78a8"])
ax.set_title(f"Cold start de ítems: Recall@10 dentro de {len(cold_items)} estrenos"); ax.set_xlabel("Recall@10")
plt.tight_layout(); plt.show(); s.round(3)

### Cold start de usuarios: de lenguaje natural a consulta (HyDE)
Un usuario nuevo escribe *"algo de ciencia ficción de los 80 con humor"*. Dos opciones:
- **Embedding directo** de la consulta.
- **HyDE** (*Hypothetical Document Embeddings*, Gao et al., 2022): el LLM escribe primero una descripción de la película ideal (en el mismo "idioma" que los ítems) y embebemos eso. Reduce el desajuste consulta↔documento.

In [ ]:
HYDE_SYSTEM = "Eres un experto en cine. Escribe en inglés, en una línea, como una ficha de catálogo."
def hyde_query(llm, q):
    return llm.generate(f"Describe con título inventado, año y géneros la película ideal para: '{q}'. "
                        "Formato: Title (year). Genres: g1, g2.", HYDE_SYSTEM, max_tokens=80, task="hyde",
                        meta={"query": q})

queries = ["algo de ciencia ficción de los 80 con humor", "película de animación para ver con niños",
           "thriller psicológico oscuro"]
rows = []
for q in queries:
    for mode, text in [("directo", q), ("HyDE", hyde_query(llm, q))]:
        hits = text_retr.search(encoder.encode([text])[0], 5)[0]
        rows.append({"consulta": q, "modo": mode, "top-5": " | ".join(movies.title.iat[i][:28] for i, _ in hits)})
pd.set_option("display.max_colwidth", 200); pd.DataFrame(rows)

## 6 · LLM-as-judge para evaluar recomendaciones

### 💡 Por qué
Las métricas offline (módulo 02) solo premian lo que el usuario **llegó a ver**: un ítem excelente que nunca se le mostró cuenta como error. Un juez LLM puede valorar la **relevancia plausible**, la **diversidad** o la **calidad de una explicación** sin humanos. Zhang et al. (2024) mostraron que, con buen prompt, LLMs como GPT-4 alcanzan acuerdo comparable al de anotadores al evaluar explicaciones, y que **ensembles de jueces heterogéneos** mejoran la estabilidad.

### ⚠️ Sesgos conocidos de los jueces
- **Posición**: prefieren la primera (o la segunda) opción → evaluar en **ambos órdenes** y contar como empate si cambian de opinión.
- **Verbosidad / autopreferencia**: prefieren respuestas largas o de su propio estilo.
- **Popularidad**: conocen mejor los títulos famosos.
- **Meta-evaluación obligatoria**: mide el acuerdo del juez con una señal de verdad (aquí, qué lista tiene mayor NDCG con el futuro real) antes de fiarte.

In [ ]:
JUDGE_SYSTEM = "Eres un evaluador imparcial de recomendaciones. Respondes SOLO con JSON."
def profile_text(u, n=12):
    liked = [i for i in hist_by_user[u] if rating_lookup.get((u, i), 0) >= 4][-n:]
    return "; ".join(short_title(i) for i in liked)

def judge_prompt(u, la, lb):
    fmt = lambda l: "\n".join(f"  {j + 1}. {short_title(i)}" for j, i in enumerate(l))
    return (f"Películas que le encantaron al usuario: {profile_text(u)}\n\nLista A:\n{fmt(la)}\n\nLista B:\n{fmt(lb)}\n\n"
            "¿Qué lista se ajusta mejor a sus gustos? Responde JSON: "
            "{\"ganador\": \"A\"|\"B\"|\"empate\", \"confianza\": 1-5, \"motivo\": \"<máx. 20 palabras>\"}")

def judge_pair(llm, u, la, lb):
    """Evalúa en ambos órdenes. Devuelve 'X' (gana la), 'Y' (gana lb) o 'empate' (incluye inconsistencias)."""
    outs = []
    for a, b in [(la, lb), (lb, la)]:
        d = parse_json(llm.generate(judge_prompt(u, a, b), JUDGE_SYSTEM, max_tokens=120, task="judge",
                                    meta={"hist_ids": hist_by_user[u][-12:], "list_a": a, "list_b": b})) or {}
        outs.append(d.get("ganador", "empate"))
    first = {"A": "X", "B": "Y"}.get(outs[0], "empate"); second = {"A": "Y", "B": "X"}.get(outs[1], "empate")
    return (first if first == second else "empate"), first == second

In [ ]:
users_j = eval_users[: (20 if FAST_DEV_RUN else 150)]
rec_sets = {"CF": cf.recommend(users_j, K), "Contenido": recommend_content(users_j, K), "Popularidad": recommend_pop(users_j, K),
            "Azar": {u: list(np.random.default_rng(u).choice(n_items, K, replace=False)) for u in users_j}}
pairs = [("CF", "Azar"), ("CF", "Popularidad"), ("CF", "Contenido")]
jrows = []
for a, b in pairs:
    for u in users_j:
        verdict, consistent = judge_pair(llm, u, rec_sets[a][u], rec_sets[b][u])
        na, nb_ = ndcg_at_k(rec_sets[a][u], truth[u]), ndcg_at_k(rec_sets[b][u], truth[u])
        jrows.append({"par": f"{a} vs {b}", "juez": verdict, "consistente": consistent,
                      "ndcg": "X" if na > nb_ else "Y" if nb_ > na else "empate"})
jd = pd.DataFrame(jrows)
wins = jd.groupby("par").juez.value_counts(normalize=True).unstack().reindex(columns=["X", "empate", "Y"]).fillna(0)
ax = wins.plot.barh(stacked=True, figsize=(8, 2.8), color=["#4c78a8", "#d3d3d3", "#e45756"])
ax.legend(["gana el 1º", "empate/inconsistente", "gana el 2º"], fontsize=8, loc="lower right")
ax.set_title("Veredictos del LLM-as-judge (evaluado en ambos órdenes)"); plt.tight_layout(); plt.show()
dec = jd[(jd.ndcg != "empate") & (jd.juez != "empate")]
print(f"Consistencia al intercambiar posiciones: {jd.consistente.mean():.0%}")
print(f"Acuerdo juez–NDCG (casos decididos por ambos, n={len(dec)}): {(dec.juez == dec.ndcg).mean():.0%}")

Interpretación: el juez debería preferir claramente CF frente a Azar (sanity check). Frente a Popularidad o Contenido el acuerdo con NDCG es la cifra interesante: **un acuerdo bajo no implica que el juez esté mal** — NDCG con un único futuro observado es una señal muy ruidosa. En producción se calibra al juez contra etiquetas humanas o contra resultados de A/B (módulo 15).

## 7 · Explicaciones generadas (y cómo cazar alucinaciones)

Spotify ha descrito el uso de LLMs (Llama ajustado) para generar **explicaciones contextualizadas** de recomendaciones y narrativas del AI DJ, con revisión de editores y tests adversariales. El riesgo: el LLM explica con datos que **no** están en la evidencia ("porque viste *X*", cuando no la viste).

Patrón robusto: **(1) evidencia determinista** calculada por el sistema (ítems del historial más parecidos, géneros compartidos) → **(2) el LLM solo redacta** → **(3) verificación automática**: todo título citado entre comillas debe estar en la evidencia.

In [ ]:
EXPLAIN_SYSTEM = ("Explicas recomendaciones en español en 1-2 frases. Solo puedes citar, entre comillas, títulos que "
                  "aparezcan en la EVIDENCIA. No inventes datos.")
def build_evidence(u, item, n=2):
    hist = hist_by_user[u][-50:]
    sims = E_text[hist] @ E_text[item]
    top = [hist[j] for j in np.argsort(-sims)[:n]]
    shared = sorted(set(movies.genre_list.iat[item]) & set(g for i in top for g in movies.genre_list.iat[i]))
    return {"titles": [short_title(i) for i in top], "genres": shared}

def explain(llm, u, item):
    ev = build_evidence(u, item)
    prompt = (f"Recomendación: {short_title(item)}\nEVIDENCIA: vio y valoró {', '.join(ev['titles'])}; "
              f"géneros en común: {', '.join(ev['genres']) or 'ninguno'}.\nEscribe la explicación.")
    return llm.generate(prompt, EXPLAIN_SYSTEM, max_tokens=120, task="explain",
                        meta={"item_id": item, "evidence_titles": ev["titles"]}), ev

catalog_titles = set(movies.title)
def check_explanation(text, ev):
    quoted = re.findall(r"[\"“«]([^\"”»]+)[\"”»]", text)
    unsupported = [q for q in quoted if q not in ev["titles"]]
    invented = [q for q in unsupported if q not in catalog_titles]
    return {"citas": len(quoted), "no_soportadas": len(unsupported), "inventadas": len(invented)}

In [ ]:
exp_rows = []
for u in eval_users[: (25 if FAST_DEV_RUN else 200)]:
    item = cf.recommend([u], 1)[u][0]
    txt, ev = explain(llm, u, item)
    exp_rows.append({"user": u, "item": short_title(item), "explicación": txt, **check_explanation(txt, ev)})
ex = pd.DataFrame(exp_rows)
rates = pd.Series({"sin citas verificables": (ex.citas == 0).mean(),
                   "cita no soportada por la evidencia": (ex.no_soportadas > 0).mean(),
                   "cita fuera del catálogo (inventada)": (ex.inventadas > 0).mean()})
ax = rates.plot.barh(figsize=(7, 2.4), color=["#bab0ac", "#f58518", "#e45756"]); ax.set_xlim(0, 1)
ax.set_title("Auditoría automática de explicaciones"); plt.tight_layout(); plt.show()
ex[["item", "explicación", "no_soportadas"]].head(5)

## 8 · Agente conversacional recomendador con LangGraph

### 💡 Qué cambia respecto a un agente "normal"
En tu curso de LangGraph construiste agentes ReAct que deciden qué herramienta llamar. Para recomendar, el patrón que funciona en la práctica es **"LLM como controlador, recomendador como herramientas"** (Chat-REC, Gao et al. 2023; InteRecAgent, Huang et al. 2023): el LLM entiende y conversa, pero **el catálogo, el retrieval y el ranking los hace el sistema de recsys**. Tres razones:

1. **Grounding**: el LLM no conoce tu catálogo (ni su disponibilidad por país, ni los estrenos de ayer). Si genera títulos, alucina. → los títulos solo salen de herramientas.
2. **Latencia y coste**: un grafo explícito con nodos deterministas hace 1–3 llamadas al LLM por turno; un ReAct libre puede hacer 5–10.
3. **Evaluabilidad**: un grafo con trazas por nodo permite tests unitarios por herramienta y evaluación de trayectorias.

### Diseño
- **Estado tipado** (`TypedDict` + *reducers* `operator.add` para mensajes, ítems mostrados y trazas).
- **Memoria a corto plazo**: el *checkpointer* (`MemorySaver`) guarda el estado por `thread_id` (preferencias de esta conversación, lo ya mostrado).
- **Memoria a largo plazo**: un `Store` (`InMemoryStore`; en producción Postgres/Redis) por usuario con lo **duradero** (p. ej. "nunca terror"), compartido entre conversaciones.
- **Herramientas**: `search` (FAISS híbrido CF+texto), `filter` (restricciones duras), `rank` (orden del retriever + re-ranking LLM), `explain` (evidencia determinista).
- **Ciclo de relajación**: si los filtros dejan < 5 candidatos, relajamos restricciones *blandas* (año, géneros pedidos) pero **nunca** las exclusiones.

In [ ]:
def draw_agent_graph():
    fig, ax = plt.subplots(figsize=(12, 3.8)); ax.axis("off"); ax.set_xlim(-0.5, 12.4); ax.set_ylim(0, 4)
    nodes = {"START": (.5, 2), "understand\n(NLU + memoria)": (2.3, 2), "retrieve\n(FAISS híbrido)": (4.6, 2.9),
             "filter\n(restricciones)": (6.9, 2.9), "relax": (6.9, 1.2), "rank\n(LLM listwise)": (9.1, 2.9),
             "respond\n(títulos del catálogo)": (11.3, 2.9), "explain\n(evidencia + auditoría)": (4.6, .8), "END": (11.3, .8)}
    for n, (x, y) in nodes.items():
        c = "#ffd6e8" if n in ("START", "END") else "#dbe9f6"
        ax.add_patch(plt.Rectangle((x - .85, y - .38), 1.7, .76, fc=c, ec="#2b6cb0", lw=1.2))
        ax.text(x, y, n, ha="center", va="center", fontsize=8)
    edges = [("START", "understand\n(NLU + memoria)", ""), ("understand\n(NLU + memoria)", "retrieve\n(FAISS híbrido)", "recomendar"),
             ("understand\n(NLU + memoria)", "explain\n(evidencia + auditoría)", "¿por qué?"),
             ("retrieve\n(FAISS híbrido)", "filter\n(restricciones)", ""), ("filter\n(restricciones)", "rank\n(LLM listwise)", "≥5"),
             ("filter\n(restricciones)", "relax", "<5"), ("relax", "filter\n(restricciones)", ""),
             ("rank\n(LLM listwise)", "respond\n(títulos del catálogo)", ""), ("respond\n(títulos del catálogo)", "END", ""),
             ("explain\n(evidencia + auditoría)", "END", "")]
    for a, b, lab in edges:
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        rad = .35 if (a, b) == ("relax", "filter\n(restricciones)") else 0
        ax.annotate("", (x2, y2), (x1, y1), arrowprops=dict(arrowstyle="->", lw=1.2, shrinkA=22, shrinkB=22,
                                                            connectionstyle=f"arc3,rad={rad}"))
        if lab:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2 + .12, lab, fontsize=7.5, color="#c0392b", ha="center")
    ax.text(.1, 3.75, "Grafo del agente CineMatch (estado: mensajes, prefs, candidatos, mostrados, trazas)", fontsize=11, weight="bold")
    plt.show()
draw_agent_graph()

### NLU: reglas + LLM
Las reglas cubren lo frecuente (géneros con sinónimos en español, negaciones, décadas) con latencia ~0 y sin alucinaciones; el LLM (si hay backend real) añade lo que las reglas no ven (*mood*, matices) y su salida **se valida** contra la lista de géneros reales.

In [ ]:
GENRE_SYNONYMS = {
    "Sci-Fi": ["ciencia ficción", "ciencia-ficción", "sci-fi", "futurista", "espacial"],
    "Comedy": ["comedia", "humor", "divertida", "graciosa"], "Horror": ["terror", "miedo", "horror"],
    "Animation": ["animación", "animada", "dibujos"], "Children": ["niños", "infantil", "familiar"],
    "Romance": ["romántica", "romance", "amor"], "Thriller": ["thriller", "suspense", "intriga"],
    "Drama": ["drama", "dramática"], "Action": ["acción"], "Adventure": ["aventura"],
    "Crime": ["crimen", "policiaca", "policíaca", "mafia"], "Documentary": ["documental"],
    "War": ["guerra", "bélica"], "Western": ["western", "vaqueros", "oeste"], "Musical": ["musical"],
    "Mystery": ["misterio"], "Fantasy": ["fantasía", "fantástica"], "Film-Noir": ["noir", "cine negro"]}
GENRE_ES = {g: syn[0] for g, syn in GENRE_SYNONYMS.items()}
NEG = r"(?:sin|nada de|no quiero|evita|excepto|menos|ni)\s+(?:\w+\s+){0,2}$"

def extract_preferences_rules(text: str) -> dict:
    t = text.lower()
    inc, exc = set(), set()
    for g, syns in GENRE_SYNONYMS.items():
        for s in syns:
            for m in re.finditer(re.escape(s), t):
                (exc if re.search(NEG, t[max(0, m.start() - 25):m.start()]) else inc).add(g)
    prefs = {"include_genres": sorted(inc - exc), "exclude_genres": sorted(exc)}
    if m := re.search(r"(?:de los|años)\s*'?(\d)0\b", t):
        d = int(m.group(1)); start = 1900 + 10 * d if d >= 2 else 2000 + 10 * d
        prefs["year_min"], prefs["year_max"] = start, start + 9
    if re.search(r"reciente|nueva|actual", t):
        prefs["year_min"] = 2010
    if re.search(r"clásic", t):
        prefs["year_max"] = 1980
    if m := re.search(r"(?:como|parecida a|similar a)\s+[\"“]?([^\"”,.?]+)", t):
        prefs["reference"] = m.group(1).strip()
    if re.search(r"por ?qu[eé]", t):
        intent = "explain"
    elif re.search(r"\b(más|otras|otra)\b", t) and not inc and "reference" not in prefs:
        intent = "more"
    else:
        intent = "recommend"
    prefs["replace"] = bool(re.search(r"\b(mejor|en cambio|olvida)\b", t))
    return {"intent": intent, **prefs}

In [ ]:
NLU_SYSTEM = "Extraes preferencias de cine de un mensaje. Respondes SOLO con JSON válido."
def extract_preferences(text: str, llm=None) -> dict:
    """Reglas (rápidas, deterministas) + LLM opcional para lo que las reglas no cubren."""
    p = extract_preferences_rules(text)
    if llm is not None and not isinstance(llm, MockLLM):
        d = parse_json(llm.generate(
            f"Mensaje: {text}\nGéneros válidos: {', '.join(ML_GENRES)}.\nJSON con claves include_genres (lista), "
            "exclude_genres (lista), year_min (int|null), year_max (int|null), mood (texto breve).",
            NLU_SYSTEM, max_tokens=200, task="nlu")) or {}
        valid = lambda gs: [g for g in (gs or []) if g in ML_GENRES]          # guardrail: solo géneros reales
        p["include_genres"] = sorted(set(p["include_genres"]) | set(valid(d.get("include_genres"))))
        p["exclude_genres"] = sorted(set(p["exclude_genres"]) | set(valid(d.get("exclude_genres"))))
        p["include_genres"] = [g for g in p["include_genres"] if g not in p["exclude_genres"]]
        for k in ["year_min", "year_max"]:
            if k not in p and isinstance(d.get(k), int):
                p[k] = d[k]
        if isinstance(d.get("mood"), str):
            p["mood"] = d["mood"]
    return p

def merge_prefs(old: dict, new: dict) -> dict:
    """Memoria de preferencias: exclusiones se acumulan; inclusiones se acumulan salvo 'mejor…'."""
    old = dict(old or {})
    inc = set(new.get("include_genres", [])) if new.get("replace") else set(old.get("include_genres", [])) | set(new.get("include_genres", []))
    exc = set(old.get("exclude_genres", [])) | set(new.get("exclude_genres", []))
    out = {"include_genres": sorted(inc - exc), "exclude_genres": sorted(exc)}
    for k in ["year_min", "year_max", "reference", "mood"]:
        if k in new:
            out[k] = new[k]
        elif k in old and not (new.get("replace") and k.startswith("year")):
            out[k] = old[k]
    return out

for s in ["Quiero ciencia ficción de los 80", "Mejor algo con humor, nada de terror", "¿Por qué me recomiendas la primera?"]:
    print(f"{s!r:45} → {extract_preferences_rules(s)}")

In [ ]:
# ---- Herramientas del agente: funciones puras sobre el recomendador (deterministas y testeables) ----
movie_years = movies.year.fillna(0).to_numpy()
movie_genres = movies.genre_list.tolist()

def zscore(v):
    return (v - v.mean()) / (v.std() + 1e-9)

def tool_search(query: str, user_id=None, k: int = 300, exclude=()) -> list:
    """Candidatos híbridos: CF del usuario (si existe) + similitud semántica de la consulta (FAISS)."""
    q = encoder.encode([query])[0] if query else None
    score = np.zeros(n_items, np.float32)
    if user_id is not None and user_id in hist_by_user:
        score += 0.6 * zscore(cf.user_emb([user_id])[0] @ cf.item_emb.T)
    if q is not None:
        score += 0.4 * zscore(E_text @ q)
    score[list(exclude)] = -np.inf
    if user_id is not None and user_id in hist_by_user:
        score[hist_by_user[user_id]] = -np.inf                      # no recomendar lo ya visto
    top = np.argpartition(-score, k)[:k]
    return [int(i) for i in top[np.argsort(-score[top])]]

def tool_filter(ids, include_genres=(), exclude_genres=(), year_min=None, year_max=None) -> list:
    inc, exc = set(include_genres), set(exclude_genres)
    keep = []
    for i in ids:
        g = set(movie_genres[i])
        if exc & g or (inc and not inc & g):
            continue
        if (year_min and movie_years[i] < year_min) or (year_max and movie_years[i] > year_max):
            continue
        keep.append(i)
    return keep

def tool_rank(ids, user_id=None, k: int = 5, use_llm: bool = True, n_rerank: int = 15) -> list:
    """Orden del retriever → (opcional) re-ranking listwise con el LLM sobre los n_rerank primeros."""
    head = list(ids[:n_rerank])
    if use_llm and user_id in hist_by_user and len(head) > 1:
        head = llm_rerank(llm, hist_by_user[user_id][-15:], head, n_perm=1)
    return head[:k] + [i for i in ids[n_rerank:] if i not in head][: max(0, k - len(head))]

def tool_explain(user_id, item_id) -> str:
    ev = build_evidence(user_id, item_id, n=1) if user_id in hist_by_user else {"titles": [], "genres": []}
    g = ", ".join(GENRE_ES.get(x, x) for x in movie_genres[item_id][:2])
    return f"{g}" + (f"; se parece a \"{ev['titles'][0]}\", que viste" if ev["titles"] else "")

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig

class RecState(TypedDict, total=False):
    messages: Annotated[list, operator.add]       # historial del hilo (memoria a corto plazo)
    user_id: int
    intent: str
    prefs: dict                                   # preferencias vigentes en la conversación
    candidates: list
    recommendations: list
    shown: Annotated[list, operator.add]          # ítems ya mostrados (no repetir)
    trace: Annotated[list, operator.add]          # log de herramientas → evaluación de trayectorias
    relaxed: bool

In [ ]:
def understand(state: RecState, config: RunnableConfig, *, store: BaseStore):
    text = state["messages"][-1]["content"]
    turn = extract_preferences(text, llm if AGENT_LLM_NLU else None)
    cfg = config.get("configurable", {})
    ns = ("prefs", str(state.get("user_id")))
    long_term = (store.get(ns, "profile").value if store.get(ns, "profile") else {}) if cfg.get("use_memory", True) else {}
    base = merge_prefs(long_term, state.get("prefs", {})) if cfg.get("use_memory", True) else {}
    prefs = merge_prefs(base, turn)
    if cfg.get("use_memory", True):                # memoria a largo plazo: solo lo duradero (exclusiones)
        store.put(ns, "profile", {"exclude_genres": prefs["exclude_genres"]})
    return {"intent": turn["intent"], "prefs": prefs, "relaxed": False,
            "trace": [{"tool": "understand", "intent": turn["intent"], "prefs": prefs}]}

def retrieve(state: RecState):
    p = state["prefs"]
    query = " ".join([p.get("mood", ""), p.get("reference", "")] + [GENRE_ES.get(g, g) for g in p.get("include_genres", [])]).strip()
    cands = tool_search(query, state.get("user_id"), exclude=state.get("shown", []))
    return {"candidates": cands, "trace": [{"tool": "search", "query": query, "n": len(cands)}]}

def filter_node(state: RecState):
    p = state["prefs"]
    if state.get("relaxed"):                      # relajamos lo blando; las exclusiones son DURAS
        kept = tool_filter(state["candidates"], exclude_genres=p.get("exclude_genres", []))
    else:
        kept = tool_filter(state["candidates"], p.get("include_genres", []), p.get("exclude_genres", []),
                           p.get("year_min"), p.get("year_max"))
    return {"candidates": kept, "trace": [{"tool": "filter", "n": len(kept), "relaxed": bool(state.get("relaxed"))}]}

def relax(state: RecState):
    return {"relaxed": True, "candidates": tool_search("", state.get("user_id"), exclude=state.get("shown", [])),
            "trace": [{"tool": "relax"}]}

def rank(state: RecState):
    recs = tool_rank(state["candidates"], state.get("user_id"), k=5, use_llm=AGENT_LLM_RERANK)
    return {"recommendations": recs, "shown": recs, "trace": [{"tool": "rank", "recs": recs}]}

def respond(state: RecState):
    recs = state["recommendations"]
    lines = [f"{j + 1}. {short_title(i)} — {tool_explain(state.get('user_id'), i)}" for j, i in enumerate(recs)]
    note = " (no encontré suficientes con todos tus filtros; relajé año/género)" if state.get("relaxed") else ""
    msg = f"Te propongo{note}:\n" + "\n".join(lines)          # títulos SIEMPRE del catálogo: el LLM no los escribe
    return {"messages": [{"role": "assistant", "content": msg}]}

def explain_node(state: RecState):
    text = state["messages"][-1]["content"].lower()
    recs = state.get("recommendations") or []
    if not recs:
        return {"messages": [{"role": "assistant", "content": "Aún no te he recomendado nada. ¿Qué te apetece ver?"}]}
    ordinals = {"primera": 0, "segunda": 1, "tercera": 2, "cuarta": 3, "quinta": 4}
    j = next((v for k, v in ordinals.items() if k in text), 0)
    txt, ev = explain(llm, state["user_id"], recs[min(j, len(recs) - 1)])
    audit = check_explanation(txt, ev)
    if audit["no_soportadas"]:                    # guardrail: si cita algo no soportado, plantilla segura
        txt = f"Comparte géneros ({', '.join(ev['genres']) or 'similares'}) con \"{ev['titles'][0]}\", que valoraste."
    return {"messages": [{"role": "assistant", "content": txt}], "trace": [{"tool": "explain", **audit}]}

In [ ]:
route_intent = lambda s: "explain" if s["intent"] == "explain" else "retrieve"
route_filter = lambda s: "relax" if len(s["candidates"]) < 5 and not s.get("relaxed") else "rank"

def build_agent():
    g = StateGraph(RecState)
    for name, fn in [("understand", understand), ("retrieve", retrieve), ("filter", filter_node),
                     ("relax", relax), ("rank", rank), ("respond", respond), ("explain", explain_node)]:
        g.add_node(name, fn)
    g.add_edge(START, "understand")
    g.add_conditional_edges("understand", route_intent, {"explain": "explain", "retrieve": "retrieve"})
    g.add_edge("retrieve", "filter")
    g.add_conditional_edges("filter", route_filter, {"relax": "relax", "rank": "rank"})
    g.add_edge("relax", "filter"); g.add_edge("rank", "respond")
    g.add_edge("respond", END); g.add_edge("explain", END)
    return g.compile(checkpointer=MemorySaver(), store=InMemoryStore())

AGENT_LLM_NLU = True          # NLU con LLM (además de reglas) si el backend es real
AGENT_LLM_RERANK = True       # re-ranking listwise con LLM en la herramienta rank
agent = build_agent()

def chat(app, text, user_id, thread="demo", use_memory=True):
    cfg = {"configurable": {"thread_id": thread, "use_memory": use_memory}}
    out = app.invoke({"messages": [{"role": "user", "content": text}], "user_id": user_id}, cfg)
    return out

In [ ]:
print(agent.get_graph().draw_mermaid())       # pégalo en https://mermaid.live para verlo
try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))   # necesita red (mermaid.ink)
except Exception as e:
    print("(sin render PNG del grafo:", type(e).__name__, ")")

In [ ]:
demo_user = eval_users[1]
for msg in ["Quiero ciencia ficción de los 80", "Mejor algo con humor, nada de terror",
            "¿Por qué me recomiendas la primera?"]:
    t0 = time.perf_counter()
    out = chat(agent, msg, demo_user, thread="demo")
    print(f"👤 {msg}\n🤖 {out['messages'][-1]['content']}\n   ⏱ {time.perf_counter() - t0:.2f}s · prefs={out['prefs']}\n")
print("Traza del último turno:", out["trace"][-2:])

In [ ]:
# Memoria a largo plazo: NUEVO hilo, mismo usuario → recuerda "nada de terror"
out2 = chat(agent, "Recomiéndame algo de suspense", demo_user, thread="otra-conversacion")
print(out2["prefs"])
print(out2["messages"][-1]["content"])
viol = [short_title(i) for i in out2["recommendations"] if "Horror" in movie_genres[i]]
print("¿Viola exclusiones?", viol or "no")

### Variante ReAct con *tool calling* nativo (opcional, API)
Para comparar, el mismo conjunto de herramientas expuesto a un agente ReAct de LangGraph con Claude (`claude-sonnet-5-5`). El LLM decide el orden de las llamadas. Observa el nº de llamadas y la latencia frente al grafo explícito.

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    from langchain_core.tools import tool
    from langchain_anthropic import ChatAnthropic
    from langgraph.prebuilt import create_react_agent

    @tool
    def buscar_peliculas(consulta: str, incluir_generos: list[str] = [], excluir_generos: list[str] = [],
                         anio_min: int = 0, anio_max: int = 3000) -> str:
        """Busca películas del catálogo de CineMatch para el usuario actual. Géneros en inglés de MovieLens."""
        ids = tool_filter(tool_search(consulta, demo_user), incluir_generos, excluir_generos, anio_min, anio_max)
        return json.dumps([{"id": i, "titulo": short_title(i), "generos": movie_genres[i]} for i in ids[:15]])

    @tool
    def explicar(item_id: int) -> str:
        """Devuelve la evidencia para explicar por qué recomendamos item_id al usuario actual."""
        return tool_explain(demo_user, item_id)

    react = create_react_agent(ChatAnthropic(model=ANTHROPIC_MODEL, max_tokens=4096), [buscar_peliculas, explicar])
    t0 = time.perf_counter()
    res = react.invoke({"messages": [("user", "Quiero ciencia ficción de los 80 con humor, nada de terror. Dame 5 y explícalas.")]})
    n_tools = sum(1 for m in res["messages"] if getattr(m, "type", "") == "tool")
    print(res["messages"][-1].content[:1500])
    print(f"\n⏱ {time.perf_counter() - t0:.1f}s · llamadas a herramientas: {n_tools}")
else:
    print("Define ANTHROPIC_API_KEY para ejecutar la variante ReAct.")

## 9 · Simulación de usuarios con LLMs (Agent4Rec / RecAgent)

### 💡 Idea
Los A/B tests son lentos y caros; la evaluación offline no captura la interacción. **Simuladores con agentes generativos**:
- **RecAgent** (Wang et al., 2023): hasta 1.000 agentes LLM con perfil, memoria y acciones (navegar, ver, chatear con amigos, publicar).
- **Agent4Rec** (Zhang et al., SIGIR 2024): agentes inicializados con usuarios reales de MovieLens/Steam/Amazon-Book, con módulos de **perfil** (gustos + rasgos sociales como *conformidad* o *actividad*), **memoria** (factual y emocional) y **acciones** (ver, puntuar, salir, responder entrevistas), que interactúan página a página con un recomendador. Evalúan la **fidelidad** (¿distinguen los ítems que el usuario real vio de ítems aleatorios?) y reproducen el efecto **filter bubble**.

### ⚠️ Cuidado
Un simulador es un **modelo**, no la realidad: los LLMs tienden a ser demasiado positivos, a preferir lo popular y lo que conocen. Antes de usarlo para decidir, mide su **fidelidad** con datos reales (lo hacemos abajo).

In [ ]:
SIM_SYSTEM = "Simulas a un usuario real de una plataforma de streaming. Sé fiel a sus gustos. Responde SOLO JSON."
class SimUser:
    def __init__(self, u, patience=2):
        liked = [i for i in hist_by_user[u] if rating_lookup.get((u, i), 0) >= 4] or hist_by_user[u]
        self.u, self.patience, self.memory = u, patience, []
        self.taste = E_text[liked].mean(0)
        g = collections.Counter(x for i in liked for x in movie_genres[i])
        self.profile = (f"Géneros favoritos: {', '.join(k for k, _ in g.most_common(4))}. "
                        f"Películas que le encantaron: {'; '.join(short_title(i) for i in liked[-8:])}.")

    def react(self, llm, page_ids):
        page = "\n".join(f"[{i}] {short_title(i)} ({', '.join(movie_genres[i])})" for i in page_ids)
        mem = f"Ya has visto en esta sesión: {'; '.join(self.memory[-5:])}.\n" if self.memory else ""
        d = parse_json(llm.generate(f"{self.profile}\n{mem}Te muestran:\n{page}\nPara cada id decide si la verías. "
                                    "JSON: [{\"id\": <id>, \"ver\": true|false}]", SIM_SYSTEM, max_tokens=300,
                                    task="simulate", meta={"taste": self.taste, "page_ids": list(page_ids)})) or []
        seen = {int(x["id"]) for x in d if isinstance(x, dict) and x.get("ver") and int(x.get("id", -1)) in set(page_ids)}
        self.memory += [short_title(i) for i in seen]
        return seen

def simulate_sessions(llm, policy, users, page_size=5, max_pages=3):
    rows = []
    for u in users:
        sim, recs, misses = SimUser(u), policy([u], page_size * max_pages)[u], 0
        for p in range(max_pages):
            page = recs[p * page_size:(p + 1) * page_size]
            seen = sim.react(llm, page)
            rows.append({"user": u, "page": p, "views": len(seen), "items": page})
            misses = misses + 1 if not seen else 0
            if misses >= sim.patience:               # el usuario se aburre y se va
                break
    return pd.DataFrame(rows)

In [ ]:
users_sim = eval_users[: (15 if FAST_DEV_RUN else 100)]
policies = {"Popularidad": recommend_pop, "Contenido": recommend_content, "CF (PureSVD)": lambda us, k: cf.recommend(us, k)}
sim_res = {}
for name, pol in policies.items():
    df = simulate_sessions(llm, pol, users_sim)
    sim_res[name] = {"tasa de visionado": df.views.sum() / (5 * len(df)),
                     "páginas por sesión": df.groupby("user").page.max().mean() + 1,
                     "cobertura (ítems distintos)": len({i for l in df["items"] for i in l})}
sim_df = pd.DataFrame(sim_res).T
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, col, c in zip(ax, sim_df.columns, ["#4c78a8", "#f58518", "#54a24b"]):
    a.bar(sim_df.index, sim_df[col], color=c); a.set_title(col); a.tick_params(axis="x", labelsize=8)
plt.suptitle("Evaluación en bucle con usuarios simulados"); plt.tight_layout(); plt.show(); sim_df

In [ ]:
# Fidelidad (estilo Agent4Rec): ¿el simulador elige los ítems que el usuario REAL vio después?
fid = []
for u in users_sim:
    pos = list(truth[u])[:2]
    neg = list(np.random.default_rng(u).choice([i for i in range(n_items) if i not in truth[u]], 3, replace=False))
    page = list(np.random.default_rng(u + 1).permutation(pos + neg))
    seen = SimUser(u).react(llm, page)
    fid += [{"real": True, "elegido": i in seen} for i in pos] + [{"real": False, "elegido": i in seen} for i in neg]
fd = pd.DataFrame(fid).groupby("real").elegido.mean()
ax = fd.rename({True: "ítems que el usuario vio", False: "ítems aleatorios"}).plot.bar(rot=0, figsize=(5, 3), color=["#bab0ac", "#4c78a8"])
ax.set_ylabel("P(el simulador lo 've')"); ax.set_title("Fidelidad del simulador"); plt.tight_layout(); plt.show(); fd

## 10 · Evaluación de agentes recomendadores

Un agente conversacional se evalúa en varias dimensiones a la vez (inspirado en iEvaLM, Wang et al. 2023, que usa usuarios simulados por LLM que conocen el ítem objetivo y conversan con el sistema):

| Dimensión | Métrica | Cómo |
|---|---|---|
| Éxito de la tarea | **Success@T**: ¿aparece el ítem objetivo en las recomendaciones antes del turno $T$? | simulador con objetivo oculto |
| Eficiencia | turnos medios hasta el éxito | ídem |
| Cumplimiento de restricciones | % de recomendaciones que violan una exclusión | comprobación determinista |
| Grounding | % de ítems fuera del catálogo / alucinados | comprobación determinista |
| Calidad de la respuesta | LLM-as-judge con rúbrica | §6 |
| Trayectoria | herramientas llamadas, bucles, errores | trazas del estado |
| Coste / latencia | tokens, p50/p95 por turno | logs |

Nuestro simulador "buscador de objetivo" toma un ítem futuro real del usuario y lo describe **sin decir el título**, añadiendo una pista por turno: género → década → segundo género → "parecida a" una película de su historial.

In [ ]:
def seeker_utterances(u, target):
    g = [x for x in movie_genres[target] if x in GENRE_ES]
    y = movie_years[target]
    hist = hist_by_user[u][-50:]
    ref = short_title(hist[int(np.argmax(E_text[hist] @ E_text[target]))])
    utt = [f"Me apetece una película de {GENRE_ES[g[0]]}" if g else "Recomiéndame algo"]
    utt.append(f"Que sea de los {int(y) // 10 % 10}0" if y else "Algo que no sea muy antiguo")
    utt.append(f"Con toques de {GENRE_ES[g[1]]}" if len(g) > 1 else "Sorpréndeme dentro de ese estilo")
    ref_clean = re.sub(r"\s*\(\d{4}\)$", "", ref)
    utt.append(f"Algo parecida a \"{ref_clean}\"")
    return utt

def run_dialogue(app, u, target, use_memory=True, thread=None):
    succ, lat, viol = [], [], 0
    for t, msg in enumerate(seeker_utterances(u, target)):
        th = (thread or f"eval-{u}") if use_memory else f"eval-{u}-{t}"
        t0 = time.perf_counter(); out = chat(app, msg, u, thread=th, use_memory=use_memory); lat.append(time.perf_counter() - t0)
        recs = out.get("recommendations", [])
        succ.append(target in recs)
        viol += sum(bool(set(movie_genres[i]) & set(out["prefs"].get("exclude_genres", []))) for i in recs)
    return {"success@turn": np.maximum.accumulate(succ), "lat": lat, "viol": viol,
            "in_catalog": all(0 <= i < n_items for i in recs)}

In [ ]:
N_DIAL = 10 if FAST_DEV_RUN else 100
dial_users = [u for u in eval_users if truth.get(u)][:N_DIAL]
results = {}
for variant, mem in [("Agente con memoria", True), ("Agente sin memoria", False)]:
    app = build_agent()                                   # agente nuevo (memorias vacías) por variante
    runs = [run_dialogue(app, u, sorted(truth[u])[0], use_memory=mem) for u in dial_users]
    results[variant] = runs
cf5 = cf.recommend(dial_users, 5)
base_curve = np.mean([sorted(truth[u])[0] in cf5[u] for u in dial_users])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for name, runs in results.items():
    ax[0].plot(range(1, 5), np.mean([r["success@turn"] for r in runs], 0), "o-", label=name)
ax[0].axhline(base_curve, ls="--", c="gray", label="CF sin conversación")
ax[0].set_xticks(range(1, 5)); ax[0].set_xlabel("turno"); ax[0].set_ylabel("Success@turno"); ax[0].set_title("¿Encuentra el agente el ítem objetivo?"); ax[0].legend(fontsize=8)
ax[1].boxplot([np.concatenate([r["lat"] for r in runs]) for runs in results.values()])
ax[1].set_xticks(range(1, len(results) + 1), list(results))
ax[1].set_ylabel("segundos por turno"); ax[1].set_title("Latencia por turno")
plt.tight_layout(); plt.show()
for name, runs in results.items():
    lat = np.concatenate([r["lat"] for r in runs])
    print(f"{name:20} violaciones de exclusión={sum(r['viol'] for r in runs)} · en catálogo={all(r['in_catalog'] for r in runs)} "
          f"· p50={np.percentile(lat, 50):.2f}s p95={np.percentile(lat, 95):.2f}s")

Lo que muestra este experimento: la memoria de la conversación acumula restricciones (género + década + matices) y estrecha la búsqueda; sin memoria, cada turno empieza de cero y solo usa la última pista. Con un LLM real añade un **juez** que puntúe la respuesta (§6) y separa los fallos por causa: ¿el objetivo no estaba en los candidatos (retrieval), se filtró mal (NLU) o se ordenó mal (rank)? Las **trazas** del estado te dan ese diagnóstico gratis.

## 11 · Costes, latencia y alucinación en producción

Usamos los contadores reales de tokens de este notebook para estimar costes a escala. Los precios de la API vienen de `PRICES_PER_MTOK` (Claude Sonnet 5.5: 2 USD por millón de tokens de entrada y 10 USD por millón de salida a fecha de escritura; compruébalo en la página de precios). El coste por hora de GPU propia y el *throughput* son **supuestos editables**.

In [ ]:
use = llm.usage()
by_task = use.groupby("task")[["in_tok", "out_tok", "latency_s"]].mean() if len(use) else pd.DataFrame()
print("Tokens y latencia medios por tipo de llamada en ESTE notebook:"); display(by_task.round(2))
tok_in = float(by_task.loc["rank", "in_tok"]) if "rank" in by_task.index else 900.0
tok_out = float(by_task.loc["rank", "out_tok"]) if "rank" in by_task.index else 80.0
p_in, p_out = PRICES_PER_MTOK["claude-sonnet-5-5"]
cost_call = (tok_in * p_in + tok_out * p_out) / 1e6
GPU_USD_H, GPU_CALLS_H = 1.0, 6000          # SUPUESTOS: $/h de una GPU y llamadas/h con vLLM en lotes
dau = np.logspace(3, 8, 30)
strategies = {"Re-ranking LLM online (2 visitas/día)": 2 * dau * cost_call,
              "Caché por usuario y día (1 llamada/DAU)": dau * cost_call,
              "Solo conversaciones (3% DAU × 5 turnos × 3 llamadas)": .03 * dau * 15 * cost_call,
              "Self-hosted 7B, caché diaria (supuesto)": np.ceil(dau / (GPU_CALLS_H * 24)) * 24 * GPU_USD_H}
fig, ax = plt.subplots(figsize=(8, 4))
for k, v in strategies.items():
    ax.loglog(dau, v, label=k)
ax.axhline(20_000 * cost_call / 30, ls=":", c="k", label="Enriquecer 20k ítems (one-off, prorrateado 30 días)")
ax.set_xlabel("usuarios activos diarios (DAU)"); ax.set_ylabel("USD / día"); ax.legend(fontsize=7.5)
ax.set_title(f"Coste diario estimado · {tok_in:.0f} tok entrada + {tok_out:.0f} salida por llamada"); plt.show()

In [ ]:
# Presupuesto de latencia ilustrativo de una home vs. un turno conversacional (ms)
budget = pd.DataFrame({"Home clásica": [10, 15, 40, 10, 0, 0], "Home + LLM rerank": [10, 15, 40, 10, 900, 0],
                       "Turno conversacional": [10, 15, 40, 10, 900, 600]},
                      index=["retrieval ANN", "features", "ranker", "reglas", "LLM rerank", "LLM respuesta (streaming)"]).T
ax = budget.plot.barh(stacked=True, figsize=(9, 2.8), colormap="tab20c")
ax.axvline(200, c="r", ls="--"); ax.text(205, 2.3, "presupuesto típico de una home", color="r", fontsize=8)
ax.set_xlabel("ms (valores ilustrativos; mide los tuyos con la celda anterior)"); ax.legend(fontsize=7, ncol=3, loc="lower right")
plt.tight_layout(); plt.show()

### Alucinación: taxonomía y defensas
| Tipo | Ejemplo | Defensa |
|---|---|---|
| Ítem inexistente | recomienda "Matrix 5" | los títulos **solo** salen de herramientas; validación de IDs |
| Ítem no disponible | existe pero no en tu país/plan | filtro duro de disponibilidad *después* del LLM |
| Explicación infiel | "porque viste X" (no la vio) | evidencia determinista + auditoría de citas (§7) |
| Formato inválido | JSON roto, índices fuera de rango | parsers defensivos + *structured outputs* + reintento |
| Inyección de prompt | una sinopsis o reseña dice "ignora las instrucciones…" | delimitar y sanear metadatos; herramientas sin efectos laterales; nunca ejecutar texto de ítems |

Para generación *restringida al catálogo* existen dos caminos: **decodificación con restricciones** (un *trie* de títulos/IDs válidos que enmascara los logits) o **semantic IDs** (módulo 11), donde el vocabulario del modelo *es* el catálogo.

## 🏭 En producción

- **Spotify** describió en 2024 (con Meta) cómo adapta **Llama** para generar explicaciones contextualizadas de recomendaciones y comentarios del **AI DJ**, con *fine-tuning* adaptado al dominio, *instruction tuning*, ingeniería de prompts y revisión de editores y tests adversariales. El AI DJ (2023) usó tecnología de OpenAI más el conocimiento de sus editores, y **AI Playlist** (2024) interpreta prompts de texto con LLMs. En 2025 Spotify ha presentado un enfoque de modelos de lenguaje abiertos combinados con embeddings de usuario y *semantic IDs* del catálogo para productos como Discover Weekly, DJ o *Prompted Playlist*.
- **Netflix** publicó (2025) su **foundation model** para recomendación: un transformer entrenado con objetivos tipo *next-token* sobre cientos de miles de millones de interacciones, que centraliza el aprendizaje de preferencias y se consume como embeddings, como subgrafo de otros modelos o con *fine-tuning*. En 2026 describió **GenRec**, un ranker basado en un LLM post-entrenado que verbaliza historial, metadatos y contexto ("context engineering").
- **YouTube / Google DeepMind** presentó **PLUM** (2025): adaptar LLMs preentrenados a recomendación generativa a escala de YouTube con *semantic IDs*, preentrenamiento continuo en datos del dominio y *fine-tuning* para retrieval.
- **LinkedIn** presentó **360Brew** (2025): un modelo decoder-only de 150B parámetros con interfaz textual que aborda más de 30 tareas de ranking y recomendación, sustituyendo modelos especializados y gran parte de la ingeniería de features.
- **Amazon** lanzó **Rufus** (2024), asistente de compra conversacional con un LLM especializado entrenado sobre catálogo, reseñas y Q&A, que también da recomendaciones.

Patrón común: el LLM se usa sobre todo **offline o como modelo fundacional del que se destilan embeddings/rankers**, y online solo donde la interacción en lenguaje natural aporta (conversación, explicaciones), con fuerte *grounding* en el catálogo.

## 🧠 Secretos de la élite

1. **Baraja siempre los candidatos.** Los rankers LLM listwise tienen sesgo de posición y de popularidad (Hou et al., 2024): reporta la media sobre varias permutaciones, o usa Borda. Un resultado con un único orden no es reproducible.
2. **Desconfía del protocolo "1 positivo + N negativos aleatorios".** Infla mucho los resultados de los LLM-rankers; evalúa re-ordenando la salida real de tu retriever y reporta el techo (Recall@K del retriever).
3. **Comprueba la memorización.** Los LLMs han visto MovieLens: Di Palma et al. (SIGIR 2025) muestran que GPT y Llama memorizan atributos e interacciones de MovieLens-1M y que el rendimiento en recomendación se relaciona con esa memorización. Valida con datos propios o anonimizando títulos.
4. **Los LLMs ganan donde faltan datos, no donde sobran.** En usuarios/ítems *warm* un EASE o SASRec bien ajustado suele ganar a un LLM zero-shot; el valor está en cold start, dominios nuevos, few-shot (TALLRec) y lenguaje natural.
5. **Pon el LLM fuera del camino crítico.** Enriquecer el catálogo cuesta $O(|\mathcal{I}|)$ una vez; re-rankear cuesta $O(\text{QPS})$ para siempre. Lo habitual es usar el LLM como *teacher* y destilar en un ranker pequeño, o cachear por (usuario, día).
6. **Meta-evalúa a tus jueces.** Un LLM-as-judge sin medir acuerdo con humanos o con A/B es una métrica sin calibrar. Evalúa en ambos órdenes y usa ensembles de jueces (Zhang et al., 2024).
7. **Los metadatos son un vector de ataque.** Títulos, sinopsis y reseñas generadas por usuarios entran en el prompt: sanea, delimita y nunca des a esas cadenas capacidad de disparar herramientas con efectos.
8. **Las trazas por nodo son tu mejor métrica de depuración.** Separar fallos de retrieval, NLU y ranking en un agente vale más que cualquier métrica agregada.

## ⚠️ Errores comunes
- Pedir al LLM que **genere títulos** libremente y luego buscarlos por *fuzzy matching* → alucinaciones y coste de *matching*.
- No fijar `temperature=0` / `do_sample=False` en ranking y evaluación → resultados no reproducibles.
- Comparar un LLM con 40 usuarios y declarar victoria sin intervalos de confianza.
- *Leakage* en TALLRec: separar por interacción en vez de por usuario, o usar ratings futuros en el historial.
- Ignorar el coste de los **tokens de entrada**: el historial y los candidatos dominan el prompt; 20 candidatos × 15 tokens + 30 títulos de historial ≈ 1.000 tokens por llamada.
- Usar memoria a largo plazo sin caducidad ni opción de borrado (privacidad, RGPD): el usuario debe poder ver y borrar lo que el agente "recuerda".
- Confiar en un simulador LLM sin medir su fidelidad frente a datos reales.

## 📝 Autoevaluación

1. ¿Por qué un embedding de texto resuelve el cold start de ítems y uno de ID no? ¿En qué caso esperas que el CF gane?
<details><summary>Respuesta</summary>El embedding de ID se aprende de interacciones: un ítem sin interacciones tiene un vector sin entrenar (o nulo). El de texto se calcula a partir de metadatos y existe desde el día 0. Para ítems y usuarios con muchas interacciones el CF captura el gusto colectivo (co-consumo) que el texto no contiene, y suele ganar.</details>

2. Describe dos sesgos de los LLM-rankers listwise y cómo mitigarlos.
<details><summary>Respuesta</summary>Sesgo de posición (los candidatos al principio o al final del prompt se favorecen) y sesgo de popularidad (prefieren títulos conocidos). Mitigación: bootstrapping (barajar B veces y agregar con Borda), prompts que enfatizan la recencia, calibración con un prior de popularidad, o fine-tuning.</details>

3. En LoRA, ¿por qué se inicializa $B=0$? ¿Cuántos parámetros entrena una capa $4096 \times 4096$ con $r=16$?
<details><summary>Respuesta</summary>Para que al inicio $W=W_0$ y el modelo arranque exactamente como el preentrenado (el entrenamiento parte de un punto estable). Parámetros: $r(d+k) = 16 \cdot 8192 = 131.072$ (0,78 % de 16,8 M).</details>

4. ¿Qué es un juez "inconsistente" y cómo lo detectas?
<details><summary>Respuesta</summary>Aquel cuyo veredicto cambia al intercambiar el orden de las opciones (sesgo de posición). Se detecta evaluando cada par en ambos órdenes; los casos inconsistentes se cuentan como empate y se reporta la tasa de consistencia.</details>

5. En el agente, ¿por qué las exclusiones son restricciones duras y el año una blanda?
<details><summary>Respuesta</summary>Violar una exclusión explícita ("nada de terror") destruye la confianza y puede ser un problema de seguridad (contenido inadecuado para niños). El año es una preferencia: si deja menos de 5 resultados es mejor relajarla y avisar que devolver una lista vacía.</details>

6. ¿Cómo medirías la fidelidad de un simulador de usuarios con LLM?
<details><summary>Respuesta</summary>Comparando sus decisiones con comportamiento real retenido: p. ej., mostrarle páginas con ítems que el usuario real consumió después mezclados con ítems aleatorios y medir si los distingue (precisión/AUC), o comparar distribuciones de ratings, longitud de sesión y popularidad consumida con los logs.</details>

7. Tienes 50 M de DAU. ¿Pondrías un LLM de re-ranking en cada carga de la home? Razona con números.
<details><summary>Respuesta</summary>Con ~1.000 tokens de entrada y ~80 de salida por llamada a 2/10 USD por millón, cada llamada cuesta ≈ 0,0028 USD; 2 cargas/día × 50 M ≈ 280.000 USD/día, más ~1 s de latencia extra que rompe el presupuesto de una home. Mejor: LLM offline (enriquecimiento/embeddings/destilación), caché por usuario-día, o solo en superficies conversacionales.</details>

## 📚 Referencias

**Surveys y marcos**
- Wu, L. et al. (2023). *A Survey on Large Language Models for Recommendation*. [arXiv:2305.19860](https://arxiv.org/abs/2305.19860)
- Lin, J. et al. (2023). *How Can Recommender Systems Benefit from Large Language Models: A Survey*. [arXiv:2306.05817](https://arxiv.org/abs/2306.05817)
- Li, L. et al. (2024). *Large Language Models for Generative Recommendation: A Survey and Visionary Discussions* (LREC-COLING 2024). [arXiv:2309.01157](https://arxiv.org/abs/2309.01157)
- AMAP/Alibaba (2025). *GR-LLMs: Recent Advances in Generative Recommendation Based on Large Language Models*. [arXiv:2507.06507](https://arxiv.org/abs/2507.06507)

**LLM como ranker / fine-tuning**
- Hou, Y. et al. (2024). *Large Language Models are Zero-Shot Rankers for Recommender Systems* (ECIR 2024). [arXiv:2305.08845](https://arxiv.org/abs/2305.08845)
- Liu, J. et al. (2023). *Is ChatGPT a Good Recommender? A Preliminary Study*. [arXiv:2304.10149](https://arxiv.org/abs/2304.10149)
- Sun, W. et al. (2023). *Is ChatGPT Good at Search? Investigating LLMs as Re-Ranking Agents* (RankGPT, EMNLP 2023). [arXiv:2304.09542](https://arxiv.org/abs/2304.09542)
- Geng, S. et al. (2022). *Recommendation as Language Processing (RLP): P5* (RecSys 2022). [arXiv:2203.13366](https://arxiv.org/abs/2203.13366)
- Bao, K. et al. (2023). *TALLRec: An Effective and Efficient Tuning Framework to Align LLM with Recommendation* (RecSys 2023). [arXiv:2305.00447](https://arxiv.org/abs/2305.00447)
- Hu, E. et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685) · Dettmers, T. et al. (2023). *QLoRA*. [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)
- Di Palma, D. et al. (2025). *Do LLMs Memorize Recommendation Datasets? A Preliminary Study on MovieLens-1M* (SIGIR 2025). [arXiv:2505.10212](https://arxiv.org/abs/2505.10212)

**Enriquecimiento, juez, explicaciones**
- Xi, Y. et al. (2023). *Towards Open-World Recommendation with Knowledge Augmentation from LLMs* (KAR). [arXiv:2306.10933](https://arxiv.org/abs/2306.10933)
- Wei, W. et al. (2024). *LLMRec: Large Language Models with Graph Augmentation for Recommendation* (WSDM 2024). [arXiv:2311.00423](https://arxiv.org/abs/2311.00423)
- Gao, L. et al. (2022). *Precise Zero-Shot Dense Retrieval without Relevance Labels* (HyDE). [arXiv:2212.10496](https://arxiv.org/abs/2212.10496)
- Zhang, X. et al. (2024). *Large Language Models as Evaluators for Recommendation Explanations*. [arXiv:2406.03248](https://arxiv.org/abs/2406.03248)

**Agentes y simulación**
- Gao, Y. et al. (2023). *Chat-REC: Towards Interactive and Explainable LLMs-Augmented Recommender System*. [arXiv:2303.14524](https://arxiv.org/abs/2303.14524)
- Huang, X. et al. (2023). *Recommender AI Agent: Integrating LLMs for Interactive Recommendations* (InteRecAgent). [arXiv:2308.16505](https://arxiv.org/abs/2308.16505)
- Wang, X. et al. (2023). *Rethinking the Evaluation for Conversational Recommendation in the Era of LLMs* (iEvaLM, EMNLP 2023). [arXiv:2305.13112](https://arxiv.org/abs/2305.13112)
- He, Z. et al. (2023). *Large Language Models as Zero-Shot Conversational Recommenders* (CIKM 2023). [arXiv:2308.10053](https://arxiv.org/abs/2308.10053)
- Wang, L. et al. (2023). *RecAgent: A Novel Simulation Paradigm for Recommender Systems*. [arXiv:2306.02552](https://arxiv.org/abs/2306.02552) · código: [YuLan-Rec](https://github.com/RUC-GSAI/YuLan-Rec)
- Zhang, A. et al. (2024). *On Generative Agents in Recommendation* (Agent4Rec, SIGIR 2024). [arXiv:2310.10108](https://arxiv.org/abs/2310.10108) · código: [Agent4Rec](https://github.com/LehengTHU/Agent4Rec)

**Industria**
- Meta AI (2024). *How Spotify is using Llama to provide personalized recommendations*. [ai.meta.com](https://ai.meta.com/blog/spotify-personalized-recommendations-built-with-llama/)
- Spotify Research (2024). *Contextualized Recommendations Through Personalized Narratives using LLMs*. research.atspotify.com
- Verma, S. (2025). *Personalization in the era of LLMs* (Spotify, AI Engineer World's Fair). [ai.engineer](https://ai.engineer/talks/personalization-in-the-era-of-llms)
- Netflix TechBlog (2025). *Foundation Model for Personalized Recommendation*. netflixtechblog.com
- Netflix (2026). *GenRec: Towards LLM-native recommendation at Netflix*. [arXiv:2608.10257](https://arxiv.org/abs/2608.10257)
- He, R. et al. (2025). *PLUM: Adapting Pre-trained Language Models for Industrial-scale Generative Recommendations* (YouTube). [arXiv:2510.07784](https://arxiv.org/abs/2510.07784)
- Firooz, H., Sanjabi, M. et al. (2025). *360Brew: A Decoder-only Foundation Model for Personalized Ranking and Recommendation* (LinkedIn). [arXiv:2501.16450](https://arxiv.org/abs/2501.16450)

**Librerías**: [LangGraph](https://langchain-ai.github.io/langgraph/) · [🤗 PEFT](https://github.com/huggingface/peft) · [vLLM](https://github.com/vllm-project/vllm) · [sentence-transformers](https://www.sbert.net/) · [FAISS](https://github.com/facebookresearch/faiss) · [Anthropic SDK](https://github.com/anthropics/anthropic-sdk-python)